# Download the data 

In [5]:
"""
=============================================================================
STEP 0: DOWNLOAD RETAILROCKET DATASET
=============================================================================
RetailRocket nằm trên Kaggle → cần Kaggle account.

CÁCH 1 (Recommend): Kaggle API
    pip install kaggle
    # Tạo API token: kaggle.com → Settings → Create New API Token
    # File kaggle.json sẽ download → đặt vào ~/.kaggle/kaggle.json
    kaggle datasets download -d retailrocket/ecommerce-dataset -p data/raw/
    cd data/raw && unzip ecommerce-dataset.zip

CÁCH 2: Manual download
    1. Vào https://www.kaggle.com/datasets/retailrocket/ecommerce-dataset
    2. Bấm Download (cần login)
    3. Giải nén vào data/raw/

SAU KHI DOWNLOAD, folder data/raw/ phải có:
    data/raw/events.csv                    (~115MB, 2.7M rows)
    data/raw/item_properties_part1.csv     (~400MB)
    data/raw/item_properties_part2.csv     (~400MB)
    data/raw/category_tree.csv             (~30KB)

CHẠY FILE NÀY ĐỂ VERIFY:
    python step0_download_retailrocket.py
=============================================================================
"""

import os
import subprocess
import sys

RAW_DIR = "data/raw"
PROCESSED_DIR = "data/processed"
FIG_DIR = "outputs/figures"

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

required_files = [
    os.path.join(RAW_DIR, "events.csv"),
    os.path.join(RAW_DIR, "item_properties_part1.csv"),
    os.path.join(RAW_DIR, "item_properties_part2.csv"),
    os.path.join(RAW_DIR, "category_tree.csv"),
]

def try_kaggle_download():
    """Thử download bằng Kaggle API."""
    try:
        import kaggle
        print("📥 Downloading via Kaggle API...")
        subprocess.run([
            "kaggle", "datasets", "download",
            "-d", "retailrocket/ecommerce-dataset",
            "-p", RAW_DIR, "--unzip"
        ], check=True)
        print("✅ Download complete!")
        return True
    except ImportError:
        print("⚠️  kaggle package not installed. Run: pip install kaggle")
        return False
    except Exception as e:
        print(f"⚠️  Kaggle API error: {e}")
        return False

def verify_files():
    """Check all required files exist."""
    all_ok = True
    for f in required_files:
        if os.path.exists(f):
            size_mb = os.path.getsize(f) / 1024 / 1024
            print(f"  ✅ {f} ({size_mb:.1f} MB)")
        else:
            print(f"  ❌ MISSING: {f}")
            all_ok = False
    return all_ok

if __name__ == "__main__":
    print("=" * 60)
    print("  RetailRocket Dataset Setup")
    print("=" * 60)
    
    print("\nChecking existing files...")
    if verify_files():
        print("\n🎉 All files ready! Run: python step1_eda_retailrocket.py")
    else:
        print("\nAttempting Kaggle API download...")
        if try_kaggle_download():
            print("\nVerifying again...")
            if verify_files():
                print("\n🎉 All files ready! Run: python step1_eda_retailrocket.py")
            else:
                print("\n❌ Some files still missing after download.")
        else:
            print(f"""
╔══════════════════════════════════════════════════════════════╗
║  MANUAL DOWNLOAD REQUIRED                                    ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  1. Go to: https://kaggle.com/datasets/retailrocket/        ║
║            ecommerce-dataset                                 ║
║  2. Click "Download" (need Kaggle account)                   ║
║  3. Unzip into: {RAW_DIR}/                            ║
║  4. Re-run this script to verify                             ║
║                                                              ║
║  OR install Kaggle API:                                      ║
║    pip install kaggle                                        ║
║    # Put kaggle.json in ~/.kaggle/                           ║
║    # Re-run this script                                      ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
""")

  RetailRocket Dataset Setup

Checking existing files...
  ✅ data/raw\events.csv (89.9 MB)
  ✅ data/raw\item_properties_part1.csv (461.9 MB)
  ✅ data/raw\item_properties_part2.csv (390.0 MB)
  ✅ data/raw\category_tree.csv (0.0 MB)

🎉 All files ready! Run: python step1_eda_retailrocket.py


We have 3 types of file: 
- events: events behavior of users
- item properties: information of each item
- category_tree: item and its parrent

# Step 0: Data Loader

In [6]:
# Load data 
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
from datetime import datetime

# ============================================================
# CONFIG
# ============================================================
RAW_DIR = "data/raw"
FIG_DIR = "outputs/figures"
STATS_DIR = "data/processed"

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(STATS_DIR, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

all_stats = {}

def section_header(title):
    print(f"\n{'='*70}")
    print(f"  {title}")
    print(f"{'='*70}")

def pct(part, whole):
    return round(part / max(whole, 1) * 100, 2)


# ============================================================
# 1. LOAD DATA
# ============================================================
section_header("1. LOAD DATA")

# --- Events (main interaction data) ---
print("Loading events.csv...")
events = pd.read_csv(
    os.path.join(RAW_DIR, "events.csv"),
    dtype={'visitorid': str, 'itemid': str, 'event': str, 'transactionid': str}
)
print(f"  Events: {len(events):,} rows × {len(events.columns)} columns")
print(f"  Columns: {list(events.columns)}")
print(f"  Dtypes:\n{events.dtypes.to_string()}")
print(f"\n  Sample (5 rows):")
print(events.head(5).to_string())

print(f"\n  Nulls:\n{events.isnull().sum().to_string()}")

# --- Category Tree ---
print("\nLoading category_tree.csv...")
cat_tree = pd.read_csv(os.path.join(RAW_DIR, "category_tree.csv"))
print(f"  Category tree: {len(cat_tree):,} rows")
print(f"  Columns: {list(cat_tree.columns)}")
print(f"  Sample:\n{cat_tree.head(5).to_string()}")

# --- Item Properties (large — load carefully) ---
print("\nLoading item_properties (may take 1-2 min)...")
props1 = pd.read_csv(os.path.join(RAW_DIR, "item_properties_part1.csv"),
                       dtype={'itemid': str, 'property': str, 'value': str})
props2 = pd.read_csv(os.path.join(RAW_DIR, "item_properties_part2.csv"),
                       dtype={'itemid': str, 'property': str, 'value': str})
item_props = pd.concat([props1, props2], ignore_index=True)
# del props1, props2
print(f"  Item properties: {len(item_props):,} rows")
print(f"  Columns: {list(item_props.columns)}")
print(f"  Sample:\n{item_props.head(5).to_string()}")

# --- Convert timestamp ---
# RetailRocket timestamp is in MILLISECONDS since epoch
events['timestamp_dt'] = pd.to_datetime(events['timestamp'], unit='ms')
events['date'] = events['timestamp_dt'].dt.date
events['hour'] = events['timestamp_dt'].dt.hour
events['day_of_week'] = events['timestamp_dt'].dt.dayofweek
events['year_month'] = events['timestamp_dt'].dt.to_period('M').astype(str)
events['week'] = events['timestamp_dt'].dt.isocalendar().week.astype(int)

print(f"\n  Time range: {events['timestamp_dt'].min()} → {events['timestamp_dt'].max()}")
time_range_days = (events['timestamp_dt'].max() - events['timestamp_dt'].min()).days
print(f"  Duration: {time_range_days} days (~{time_range_days/30:.1f} months)")




  1. LOAD DATA
Loading events.csv...
  Events: 2,756,101 rows × 5 columns
  Columns: ['timestamp', 'visitorid', 'event', 'itemid', 'transactionid']
  Dtypes:
timestamp         int64
visitorid        object
event            object
itemid           object
transactionid    object

  Sample (5 rows):
       timestamp visitorid event  itemid transactionid
0  1433221332117    257597  view  355908           NaN
1  1433224214164    992329  view  248676           NaN
2  1433221999827    111016  view  318965           NaN
3  1433221955914    483717  view  253185           NaN
4  1433221337106    951259  view  367447           NaN

  Nulls:
timestamp              0
visitorid              0
event                  0
itemid                 0
transactionid    2733644

Loading category_tree.csv...
  Category tree: 1,669 rows
  Columns: ['categoryid', 'parentid']
  Sample:
   categoryid  parentid
0        1016     213.0
1         809     169.0
2         570       9.0
3        1691     885.0
4         

# Step 2: Inital EDA for Cleaning and making some decision with data

## 2A: Data Quality check

In [ ]:
"""
STEP 2A: INITIAL EDA + DUPLICATE REMOVAL
Chỉ: nhìn data, check quality, xóa duplicates. DỪNG.
"""

import pandas as pd
import numpy as np
import os

RAW_DIR = "data/raw"
OUT_DIR = "data/processed"
os.makedirs(OUT_DIR, exist_ok=True)



# ============================================================
# 2. DATA QUALITY CHECKS
# ============================================================
print("\n" + "="*60)
print("  2. DATA QUALITY CHECKS")
print("="*60)

# --- Nulls ---
print("\n--- Null values ---")
print(events.isnull().sum().to_string())

# --- Event types ---
print("\n--- Event types ---")
event_dist = events['event'].value_counts()
for e, c in event_dist.items():
    print(f"  {e:15s}: {c:>12,} ({c/len(events)*100:.1f}%)")

expected = {'view', 'addtocart', 'transaction'}
unexpected = set(events['event'].unique()) - expected
print(f"  Unexpected types: {unexpected if unexpected else 'None ✓'}")

# --- Unique counts ---
print(f"\n--- Unique counts ---")
print(f"  Users: {events['visitorid'].nunique():,}")
print(f"  Items: {events['itemid'].nunique():,}")

# --- transactionid integrity ---
print(f"\n--- transactionid integrity ---")

# Transaction events có transactionid?
trans_events = events[events['event'] == 'transaction']
trans_missing_id = trans_events['transactionid'].isnull().sum()
print(f"  Transaction events: {len(trans_events):,}")
print(f"  Transaction events missing transactionid: {trans_missing_id:,}")

# Non-transaction events có transactionid? (không nên có)
non_trans_with_id = events[
    (events['event'] != 'transaction') & (events['transactionid'].notnull())
]
print(f"  Non-transaction rows WITH transactionid: {len(non_trans_with_id):,}")
if len(non_trans_with_id) > 0:
    print(f"    → PROBLEM: should be NaN for view/addtocart")
    print(f"    → Breakdown: {non_trans_with_id['event'].value_counts().to_dict()}")

# Transaction count vs unique IDs
n_unique_txn_ids = trans_events['transactionid'].nunique()
print(f"\n  Unique transaction IDs: {n_unique_txn_ids:,}")
print(f"  Transaction events: {len(trans_events):,}")
print(f"  Ratio: {len(trans_events)/max(n_unique_txn_ids,1):.2f}")
print(f"    → >1 means multi-item transactions (basket buying)")

# Items per transaction
trans_with_id = trans_events[trans_events['transactionid'].notnull()]
items_per_txn = trans_with_id.groupby('transactionid')['itemid'].nunique()
print(f"\n  Items per transaction:")
print(f"    Mean:   {items_per_txn.mean():.2f}")
print(f"    Median: {items_per_txn.median():.0f}")
print(f"    Max:    {items_per_txn.max()}")
print(f"    Multi-item transactions: {(items_per_txn > 1).sum():,} ({(items_per_txn > 1).mean()*100:.1f}%)")

# --- Empty/invalid IDs ---
print(f"\n--- Empty/invalid values ---")
print(f"  Empty visitorid: {(events['visitorid'] == '').sum():,}")
print(f"  Empty itemid: {(events['itemid'] == '').sum():,}")
print(f"  Negative timestamps: {(events['timestamp'] < 0).sum():,}")

# --- Duplicates ---
print(f"\n--- Duplicates ---")

# Full row duplicates
full_dups = events.duplicated().sum()
print(f"  Full row duplicates: {full_dups:,}")

# Same (timestamp, user, item, event) — true duplicates
event_dups = events.duplicated(subset=['timestamp', 'visitorid', 'itemid', 'event']).sum()
print(f"  Event duplicates (same ts+user+item+event): {event_dups:,}")

# Same (user, item, event) different time — NOT duplicates, repeat visits
repeat = events.groupby(['visitorid', 'itemid', 'event']).size()
repeat_multi = (repeat > 1).sum()
print(f"  Repeat interactions (same user+item+event, different time): {repeat_multi:,}")
print(f"    → These are NOT duplicates — user revisits")


# ============================================================
# 3. FIX QUALITY + REMOVE DUPLICATES
# ============================================================
print("\n" + "="*60)
print("  3. FIX + REMOVE DUPLICATES")
print("="*60)

rows_start = len(events)

# Fix: transactionid trên non-transaction events → NaN
n_fixed = len(events[(events['event'] != 'transaction') & (events['transactionid'].notnull())])
events.loc[events['event'] != 'transaction', 'transactionid'] = np.nan
print(f"\nFixed transactionid on non-transaction rows: {n_fixed:,} → set NaN")

# Remove exact event duplicates (same timestamp+user+item+event)
before = len(events)
events = events.drop_duplicates(subset=['timestamp', 'visitorid', 'itemid', 'event'])
after = len(events)
removed = before - after
print(f"\nRemoved duplicates: {removed:,} ({removed/before*100:.2f}%)")
print(f"  Before: {before:,}")
print(f"  After:  {after:,}")

# ============================================================
# 4. POST-CLEAN SUMMARY
# ============================================================
print("\n" + "="*60)
print("  4. POST-CLEAN SUMMARY (before bot removal)")
print("="*60)

print(f"\n  Events: {len(events):,} (was {rows_start:,}, removed {rows_start - len(events):,})")
print(f"  Users:  {events['visitorid'].nunique():,}")
print(f"  Items:  {events['itemid'].nunique():,}")

print(f"\n  Event distribution (post-dedup):")
for e, c in events['event'].value_counts().items():
    print(f"    {e:15s}: {c:>12,} ({c/len(events)*100:.1f}%)")

# Save deduped (NOT bot-cleaned yet)
events.to_parquet(os.path.join(OUT_DIR, "events_deduped.parquet"), index=False)
print(f"\n💾 Saved: {OUT_DIR}/events_deduped.parquet")



  2. DATA QUALITY CHECKS

--- Null values ---
timestamp              0
visitorid              0
event                  0
itemid                 0
transactionid    2733644
timestamp_dt           0
date                   0
hour                   0
day_of_week            0
year_month             0
week                   0

--- Event types ---
  view           :    2,664,312 (96.7%)
  addtocart      :       69,332 (2.5%)
  transaction    :       22,457 (0.8%)
  Unexpected types: None ✓

--- Unique counts ---
  Users: 1,407,580
  Items: 235,061

--- transactionid integrity ---
  Transaction events: 22,457
  Transaction events missing transactionid: 0
  Non-transaction rows WITH transactionid: 0

  Unique transaction IDs: 17,672
  Transaction events: 22,457
  Ratio: 1.27
    → >1 means multi-item transactions (basket buying)

  Items per transaction:
    Mean:   1.26
    Median: 1
    Max:    31
    Multi-item transactions: 2,643 (15.0%)

--- Empty/invalid values ---
  Empty visitorid: 0
  

## 2B: Bot detection

In [ ]:
"""
STEP 2B: BOT DETECTION
Assumes: events DataFrame already in memory (from Step 1A)
Output: 5 charts + tables → chọn threshold → filter ở cell cuối
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

FIG_DIR = "outputs/figures"
os.makedirs(FIG_DIR, exist_ok=True)

# Đảm bảo có columns cần thiết
if 'timestamp_dt' not in events.columns:
    events['timestamp_dt'] = pd.to_datetime(events['timestamp'], unit='ms')
if 'date' not in events.columns:
    events['date'] = events['timestamp_dt'].dt.date

print(f"Working with: {len(events):,} events, {events['visitorid'].nunique():,} users")


# ============================================================
# PHÂN TÍCH 1: Daily Activity ECDF
# ============================================================
print("\n" + "="*60)
print("  PHÂN TÍCH 1: Daily Activity ECDF")
print("="*60)

daily_activity = events.groupby(['visitorid', 'date']).size().reset_index(name='daily_count')
user_max_daily = daily_activity.groupby('visitorid')['daily_count'].max().reset_index(name='max_daily')

# ECDF
sorted_vals = np.sort(user_max_daily['max_daily'].values)
ecdf_y = np.arange(1, len(sorted_vals) + 1) / len(sorted_vals)

# Print percentiles
print("Max daily events per user (percentiles):")
for p, q in [('P90', 0.9), ('P95', 0.95), ('P99', 0.99), ('P99.5', 0.995), ('P99.9', 0.999)]:
    val = np.percentile(sorted_vals, float(q * 100))
    print(f"  {p:6s}: {val:>6.0f} events/day")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Full ECDF
axes[0].plot(sorted_vals, ecdf_y, linewidth=1.5, color='steelblue')
axes[0].set_xlabel('Max Daily Events per User', fontsize=12)
axes[0].set_ylabel('Cumulative Proportion of Users', fontsize=12)
axes[0].set_title('ECDF — Max Daily Activity (full range)', fontsize=13, fontweight='bold')
axes[0].set_xscale('log')
axes[0].axhline(y=0.99, color='red', linestyle='--', alpha=0.5, label='P99')
axes[0].axhline(y=0.999, color='orange', linestyle='--', alpha=0.5, label='P99.9')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Zoomed ECDF — top 5% (where bots live)
mask_top = ecdf_y >= 0.95
axes[1].plot(sorted_vals[mask_top], ecdf_y[mask_top], linewidth=1.5, color='steelblue')
axes[1].set_xlabel('Max Daily Events per User', fontsize=12)
axes[1].set_ylabel('Cumulative Proportion', fontsize=12)
axes[1].set_title('ECDF — Zoomed Top 5% (elbow region)', fontsize=13, fontweight='bold')
axes[1].axhline(y=0.99, color='red', linestyle='--', alpha=0.5, label='P99')
axes[1].axhline(y=0.999, color='orange', linestyle='--', alpha=0.5, label='P99.9')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/bot_01_daily_ecdf.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"💾 {FIG_DIR}/bot_01_daily_ecdf.png")


# ============================================================
# PHÂN TÍCH 2: Lifetime Activity ECDF
# ============================================================
print("\n" + "="*60)
print("  PHÂN TÍCH 2: Lifetime Activity ECDF")
print("="*60)

user_total = events.groupby('visitorid').size().reset_index(name='total_events')

sorted_total = np.sort(user_total['total_events'].values)
ecdf_total = np.arange(1, len(sorted_total) + 1) / len(sorted_total)

print("Total lifetime events per user (percentiles):")
for p, q in [('P90', 0.9), ('P95', 0.95), ('P99', 0.99), ('P99.5', 0.995), ('P99.9', 0.999)]:
    val = np.percentile(sorted_total, float(q * 100))
    print(f"  {p:6s}: {val:>6.0f} events total")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].plot(sorted_total, ecdf_total, linewidth=1.5, color='coral')
axes[0].set_xlabel('Total Lifetime Events per User', fontsize=12)
axes[0].set_ylabel('Cumulative Proportion of Users', fontsize=12)
axes[0].set_title('ECDF — Lifetime Activity (full range)', fontsize=13, fontweight='bold')
axes[0].set_xscale('log')
axes[0].axhline(y=0.99, color='red', linestyle='--', alpha=0.5, label='P99')
axes[0].axhline(y=0.999, color='orange', linestyle='--', alpha=0.5, label='P99.9')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

mask_top2 = ecdf_total >= 0.95
axes[1].plot(sorted_total[mask_top2], ecdf_total[mask_top2], linewidth=1.5, color='coral')
axes[1].set_xlabel('Total Lifetime Events', fontsize=12)
axes[1].set_ylabel('Cumulative Proportion', fontsize=12)
axes[1].set_title('ECDF — Zoomed Top 5%', fontsize=13, fontweight='bold')
axes[1].axhline(y=0.99, color='red', linestyle='--', alpha=0.5, label='P99')
axes[1].axhline(y=0.999, color='orange', linestyle='--', alpha=0.5, label='P99.9')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/bot_02_lifetime_ecdf.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"💾 {FIG_DIR}/bot_02_lifetime_ecdf.png")


# ============================================================
# PHÂN TÍCH 3: Behavior Pattern theo Activity Level
# ============================================================
print("\n" + "="*60)
print("  PHÂN TÍCH 3: Behavior Pattern by Activity Level")
print("="*60)

# Merge max daily vào user-level stats
user_stats = events.groupby('visitorid').agg(
    total_events=('event', 'count'),
    n_views=('event', lambda x: (x == 'view').sum()),
    n_carts=('event', lambda x: (x == 'addtocart').sum()),
    n_trans=('event', lambda x: (x == 'transaction').sum()),
    unique_items=('itemid', 'nunique'),
).reset_index()

user_stats = user_stats.merge(user_max_daily, on='visitorid')
user_stats['has_purchase'] = user_stats['n_trans'] > 0
user_stats['has_cart'] = user_stats['n_carts'] > 0
user_stats['purchase_rate'] = user_stats['n_trans'] / user_stats['total_events']

# Bins theo max daily events
bins = [0, 5, 10, 20, 50, 100, 200, float('inf')]
labels = ['1-5', '6-10', '11-20', '21-50', '51-100', '101-200', '200+']
user_stats['activity_bin'] = pd.cut(user_stats['max_daily'], bins=bins, labels=labels)

# Aggregate per bin
bin_stats = user_stats.groupby('activity_bin', observed=True).agg(
    n_users=('visitorid', 'count'),
    pct_has_purchase=('has_purchase', 'mean'),
    pct_has_cart=('has_cart', 'mean'),
    avg_unique_items=('unique_items', 'mean'),
    avg_total_events=('total_events', 'mean'),
    avg_purchase_rate=('purchase_rate', 'mean'),
).round(4)

bin_stats['pct_has_purchase'] = (bin_stats['pct_has_purchase'] * 100).round(1)
bin_stats['pct_has_cart'] = (bin_stats['pct_has_cart'] * 100).round(1)
bin_stats['avg_purchase_rate'] = (bin_stats['avg_purchase_rate'] * 100).round(2)

print("\nBehavior by max daily activity level:")
print(f"{'Bin':>10s} {'Users':>10s} {'%Purchase':>10s} {'%Cart':>8s} {'AvgItems':>10s} {'AvgEvents':>10s} {'PurchRate%':>10s}")
print(f"{'-'*68}")
for idx, row in bin_stats.iterrows():
    print(f"{idx:>10s} {row['n_users']:>10,} {row['pct_has_purchase']:>9.1f}% {row['pct_has_cart']:>7.1f}% "
          f"{row['avg_unique_items']:>10.1f} {row['avg_total_events']:>10.1f} {row['avg_purchase_rate']:>9.2f}%")

# Chart
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

x = range(len(bin_stats))
xlabels = bin_stats.index.tolist()

axes[0].bar(x, bin_stats['n_users'], color='steelblue', edgecolor='white')
axes[0].set_xticks(x)
axes[0].set_xticklabels(xlabels, rotation=30, ha='right')
axes[0].set_ylabel('Number of Users')
axes[0].set_title('Users per Activity Bin', fontsize=13, fontweight='bold')
axes[0].set_yscale('log')

axes[1].bar(x, bin_stats['pct_has_purchase'], color='#2ecc71', edgecolor='white')
axes[1].set_xticks(x)
axes[1].set_xticklabels(xlabels, rotation=30, ha='right')
axes[1].set_ylabel('% Users with Purchase')
axes[1].set_title('Purchase Rate by Activity', fontsize=13, fontweight='bold')

axes[2].bar(x, bin_stats['avg_purchase_rate'], color='coral', edgecolor='white')
axes[2].set_xticks(x)
axes[2].set_xticklabels(xlabels, rotation=30, ha='right')
axes[2].set_ylabel('Avg Purchase Rate %')
axes[2].set_title('Avg Transaction/Event Ratio', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/bot_03_behavior_bins.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"💾 {FIG_DIR}/bot_03_behavior_bins.png")


# ============================================================
# PHÂN TÍCH 4: Inter-Event Time cho High vs Normal Activity
# ============================================================
print("\n" + "="*60)
print("  PHÂN TÍCH 4: Inter-Event Time (High vs Normal Users)")
print("="*60)

# Tính inter-event time cho mỗi user
events_sorted = events.sort_values(['visitorid', 'timestamp_dt'])
events_sorted['prev_ts'] = events_sorted.groupby('visitorid')['timestamp_dt'].shift(1)
events_sorted['gap_sec'] = (events_sorted['timestamp_dt'] - events_sorted['prev_ts']).dt.total_seconds()
events_sorted = events_sorted.dropna(subset=['gap_sec'])

# Chỉ lấy gaps trong cùng ngày (cross-day gaps không meaningful cho bot detection)
events_sorted['prev_date'] = events_sorted.groupby('visitorid')['date'].shift(1)
same_day = events_sorted[events_sorted['date'] == events_sorted['prev_date']]

# Split: normal users (max daily ≤ 20) vs high activity (max daily > 20)
high_users = set(user_max_daily[user_max_daily['max_daily'] > 20]['visitorid'])
normal_users = set(user_max_daily[user_max_daily['max_daily'] <= 20]['visitorid'])

gaps_normal = same_day[same_day['visitorid'].isin(normal_users)]['gap_sec']
gaps_high = same_day[same_day['visitorid'].isin(high_users)]['gap_sec']

print(f"Normal users (≤20/day): {len(normal_users):,} users, {len(gaps_normal):,} gaps")
print(f"High users (>20/day):   {len(high_users):,} users, {len(gaps_high):,} gaps")

if len(gaps_normal) > 0:
    print(f"\nInter-event time (seconds) — Normal users:")
    for p, q in [('P10', 0.1), ('P25', 0.25), ('median', 0.5), ('P75', 0.75), ('P90', 0.9)]:
        val = gaps_normal.quantile(q)
        print(f"  {p:8s}: {val:>8.1f}s ({val/60:.1f} min)")

if len(gaps_high) > 0:
    print(f"\nInter-event time (seconds) — High activity users:")
    for p, q in [('P10', 0.1), ('P25', 0.25), ('median', 0.5), ('P75', 0.75), ('P90', 0.9)]:
        val = gaps_high.quantile(q)
        print(f"  {p:8s}: {val:>8.1f}s ({val/60:.1f} min)")

# Chart
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

if len(gaps_normal) > 0:
    axes[0].hist(gaps_normal.clip(upper=600), bins=100, color='steelblue', edgecolor='white', alpha=0.8)
    axes[0].set_xlabel('Seconds between events', fontsize=11)
    axes[0].set_ylabel('Count', fontsize=11)
    axes[0].set_title('Inter-Event Time — Normal Users (≤20/day)', fontsize=13, fontweight='bold')
    axes[0].axvline(x=gaps_normal.median(), color='red', linestyle='--',
                    label=f'Median: {gaps_normal.median():.0f}s')
    axes[0].legend()

if len(gaps_high) > 0:
    axes[1].hist(gaps_high.clip(upper=600), bins=100, color='coral', edgecolor='white', alpha=0.8)
    axes[1].set_xlabel('Seconds between events', fontsize=11)
    axes[1].set_ylabel('Count', fontsize=11)
    axes[1].set_title('Inter-Event Time — High Activity Users (>20/day)', fontsize=13, fontweight='bold')
    axes[1].axvline(x=gaps_high.median(), color='red', linestyle='--',
                    label=f'Median: {gaps_high.median():.0f}s')
    axes[1].legend()

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/bot_04_inter_event.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"💾 {FIG_DIR}/bot_04_inter_event.png")

# Cleanup
del events_sorted, same_day


# ============================================================
# PHÂN TÍCH 5: Scatter — Max Daily Events vs Purchase Rate
# ============================================================
print("\n" + "="*60)
print("  PHÂN TÍCH 5: Max Daily Activity vs Purchase Behavior")
print("="*60)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter: max daily vs purchase rate
# Subsample nếu quá nhiều points
plot_df = user_stats.copy()
if len(plot_df) > 50000:
    plot_df = plot_df.sample(50000, random_state=42)

axes[0].scatter(
    plot_df['max_daily'],
    plot_df['purchase_rate'] * 100,
    s=2, alpha=0.2, c='steelblue'
)
axes[0].set_xlabel('Max Daily Events', fontsize=11)
axes[0].set_ylabel('Purchase Rate (%)', fontsize=11)
axes[0].set_title('Max Daily Activity vs Purchase Rate', fontsize=13, fontweight='bold')
axes[0].set_xscale('log')
axes[0].grid(True, alpha=0.3)

# Scatter: max daily vs has_purchase (binary)
# Jitter for visibility
jitter = np.random.uniform(-0.1, 0.1, len(plot_df))
axes[1].scatter(
    plot_df['max_daily'],
    plot_df['has_purchase'].astype(int) + jitter,
    s=2, alpha=0.1, c='coral'
)
axes[1].set_xlabel('Max Daily Events', fontsize=11)
axes[1].set_ylabel('Has Purchase (0/1, jittered)', fontsize=11)
axes[1].set_title('Max Daily Activity vs Has Any Purchase', fontsize=13, fontweight='bold')
axes[1].set_xscale('log')
axes[1].set_yticks([0, 1])
axes[1].set_yticklabels(['No Purchase', 'Has Purchase'])
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/bot_05_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"💾 {FIG_DIR}/bot_05_scatter.png")





Working with: 2,755,641 events, 1,407,580 users

  PHÂN TÍCH 1: Daily Activity ECDF
Max daily events per user (percentiles):
  P90   :      3 events/day
  P95   :      4 events/day
  P99   :      9 events/day
  P99.5 :     13 events/day
  P99.9 :     27 events/day


C:\Users\NITRO V 15\AppData\Local\Temp\ipykernel_20844\321146288.py:70: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


💾 outputs/figures/bot_01_daily_ecdf.png

  PHÂN TÍCH 2: Lifetime Activity ECDF
Total lifetime events per user (percentiles):
  P90   :      3 events total
  P95   :      5 events total
  P99   :     13 events total
  P99.5 :     19 events total
  P99.9 :     47 events total


C:\Users\NITRO V 15\AppData\Local\Temp\ipykernel_20844\321146288.py:115: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


💾 outputs/figures/bot_02_lifetime_ecdf.png

  PHÂN TÍCH 3: Behavior Pattern by Activity Level

Behavior by max daily activity level:
       Bin      Users  %Purchase    %Cart   AvgItems  AvgEvents PurchRate%
--------------------------------------------------------------------
       1-5 1,367,462.0       0.5%     1.9%        1.3        1.5      0.13%
      6-10   28,650.0      10.7%    25.3%        5.3        9.3      1.61%
     11-20    8,718.0      17.4%    38.2%       10.5       19.1      1.89%
     21-50    2,385.0      26.5%    52.2%       25.7       47.2      2.18%
    51-100      236.0      38.6%    62.7%      101.0      167.4      3.02%
   101-200      100.0      70.0%    82.0%      311.8      517.2      4.81%
      200+       29.0      75.9%    89.7%      822.0     1423.7      3.50%


C:\Users\NITRO V 15\AppData\Local\Temp\ipykernel_20844\321146288.py:193: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


💾 outputs/figures/bot_03_behavior_bins.png

  PHÂN TÍCH 4: Inter-Event Time (High vs Normal Users)
Normal users (≤20/day): 1,404,830 users, 563,866 gaps
High users (>20/day):   2,750 users, 228,919 gaps

Inter-event time (seconds) — Normal users:
  P10     :     10.4s (0.2 min)
  P25     :     28.8s (0.5 min)
  median  :     82.9s (1.4 min)
  P75     :    315.6s (5.3 min)
  P90     :   2531.3s (42.2 min)

Inter-event time (seconds) — High activity users:
  P10     :      7.1s (0.1 min)
  P25     :     23.2s (0.4 min)
  median  :     73.1s (1.2 min)
  P75     :    253.3s (4.2 min)
  P90     :   1053.2s (17.6 min)


C:\Users\NITRO V 15\AppData\Local\Temp\ipykernel_20844\321146288.py:259: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


💾 outputs/figures/bot_04_inter_event.png

  PHÂN TÍCH 5: Max Daily Activity vs Purchase Behavior
💾 outputs/figures/bot_05_scatter.png

  SUMMARY — Chờ chọn threshold

Nhìn 5 charts + tables ở trên, quyết định:
  1. ECDF elbow ở đâu?
  2. Bins nào có purchase rate = 0 hoặc gần 0? → likely bots
  3. Inter-event time: high users có median < 2s? → machine speed
  4. Scatter: chỗ nào gap giữa normal vs anomaly?

Sau khi chọn threshold, chạy cell dưới để filter:



C:\Users\NITRO V 15\AppData\Local\Temp\ipykernel_20844\321146288.py:310: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:

# === CHẠY SAU KHI CHỌN THRESHOLD ===
# Thay số X bằng threshold bạn chọn

DAILY_THRESHOLD = 200  # <-- thay X

bot_user_ids = user_max_daily[user_max_daily['max_daily'] > DAILY_THRESHOLD]['visitorid']
bot_users = set(bot_user_ids)

print(f"Threshold: >{DAILY_THRESHOLD} events/day")
print(f"Bot users: {len(bot_users):,}")

before = len(events)
events_clean = events[~events['visitorid'].isin(bot_users)].copy()
after = len(events_clean)

print(f"Before: {before:,}")
print(f"After:  {after:,}")
print(f"Removed: {before - after:,} ({(before-after)/before*100:.2f}%)")
print(f"Users remaining: {events_clean['visitorid'].nunique():,}")
print(f"Items remaining: {events_clean['itemid'].nunique():,}")

# Save
events_clean.drop(columns=['timestamp_dt', 'date'], errors='ignore').to_parquet(
    "data/processed/events_bot_cleaned.parquet", index=False
)
print("💾 Saved: data/processed/events_bot_cleaned.parquet")


Threshold: >200 events/day
Bot users: 29
Before: 2,755,641
After:  2,714,354
Removed: 41,287 (1.50%)
Users remaining: 1,407,551
Items remaining: 234,151
💾 Saved: data/processed/events_bot_cleaned.parquet


## 2C: K-Core Simulation

In [ ]:
"""
STEP 2C-2: K-CORE SIMULATION
Assumes: events df đã bot-cleaned (từ Step 1C) trong memory.

Chạy K từ 2 đến 20.
Mỗi K show:
  - Events/Users/Items còn lại (%)
  - Transactions survived (% — quan trọng nhất vì đây là positive labels)
  - Buyers survived (%)
  - Density (quyết định CF feasibility)
  - Avg events/user (quyết định sequence quality)

Output: table + charts → bạn chọn K
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

events = events_clean  # rename cho rõ ràng
FIG_DIR = "outputs/figures"
os.makedirs(FIG_DIR, exist_ok=True)

# Đảm bảo có timestamp_dt
if 'timestamp_dt' not in events.columns:
    events['timestamp_dt'] = pd.to_datetime(events['timestamp'], unit='ms')

# Baseline counts
N_EVENTS = len(events)
N_USERS = events['visitorid'].nunique()
N_ITEMS = events['itemid'].nunique()
N_TRANS = (events['event'] == 'transaction').sum()
N_BUYERS = events[events['event'] == 'transaction']['visitorid'].nunique()
N_CARTS = (events['event'] == 'addtocart').sum()

print(f"Baseline (after bot removal):")
print(f"  Events:       {N_EVENTS:>12,}")
print(f"  Users:        {N_USERS:>12,}")
print(f"  Items:        {N_ITEMS:>12,}")
print(f"  Transactions: {N_TRANS:>12,}")
print(f"  Buyers:       {N_BUYERS:>12,}")

# ============================================================
# RUN K-CORE FOR K = 2..20
# ============================================================
print(f"\n{'='*90}")
print(f"  K-CORE SIMULATION")
print(f"{'='*90}")

K_VALUES = list(range(2, 21))
results = []

for k in K_VALUES:
    temp = events[['visitorid', 'itemid', 'event']].copy()
    prev_len = len(temp)
    iters = 0
    
    while True:
        # Filter items with < K events
        ic = temp.groupby('itemid').size()
        temp = temp[temp['itemid'].isin(ic[ic >= k].index)]
        
        # Filter users with < K events
        uc = temp.groupby('visitorid').size()
        temp = temp[temp['visitorid'].isin(uc[uc >= k].index)]
        
        iters += 1
        if len(temp) == prev_len:
            break
        prev_len = len(temp)
    
    n_events = len(temp)
    n_users = temp['visitorid'].nunique()
    n_items = temp['itemid'].nunique()
    n_trans = (temp['event'] == 'transaction').sum()
    n_buyers = temp[temp['event'] == 'transaction']['visitorid'].nunique()
    n_carts = (temp['event'] == 'addtocart').sum()
    density = n_events / max(n_users * n_items, 1) * 100
    avg_events = n_events / max(n_users, 1)
    
    results.append({
        'K': k,
        'events': n_events,
        'users': n_users,
        'items': n_items,
        'transactions': n_trans,
        'buyers': n_buyers,
        'carts': n_carts,
        'density': density,
        'avg_events_per_user': avg_events,
        'pct_events': n_events / N_EVENTS * 100,
        'pct_users': n_users / N_USERS * 100,
        'pct_items': n_items / N_ITEMS * 100,
        'pct_trans': n_trans / max(N_TRANS, 1) * 100,
        'pct_buyers': n_buyers / max(N_BUYERS, 1) * 100,
        'iterations': iters,
    })

res = pd.DataFrame(results)

# ============================================================
# PRINT TABLE
# ============================================================
print(f"\n{'K':>3s} {'Events':>10s} {'%Evts':>7s} {'Users':>8s} {'%Usrs':>7s} "
      f"{'Items':>8s} {'%Itms':>7s} {'Trans':>7s} {'%Trans':>7s} "
      f"{'Buyers':>7s} {'%Byrs':>7s} {'Density':>9s} {'Avg/Usr':>8s}")
print(f"{'-'*95}")

for _, r in res.iterrows():
    print(f"{r['K']:>3.0f} {r['events']:>10,.0f} {r['pct_events']:>6.1f}% "
          f"{r['users']:>8,.0f} {r['pct_users']:>6.1f}% "
          f"{r['items']:>8,.0f} {r['pct_items']:>6.1f}% "
          f"{r['transactions']:>7,.0f} {r['pct_trans']:>6.1f}% "
          f"{r['buyers']:>7,.0f} {r['pct_buyers']:>6.1f}% "
          f"{r['density']:>8.4f}% {r['avg_events_per_user']:>8.1f}")

# ============================================================
# CHARTS
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Chart 1: % remaining — events, users, items
axes[0,0].plot(res['K'], res['pct_events'], 'o-', label='Events', linewidth=2)
axes[0,0].plot(res['K'], res['pct_users'], 's-', label='Users', linewidth=2)
axes[0,0].plot(res['K'], res['pct_items'], '^-', label='Items', linewidth=2)
axes[0,0].set_xlabel('K', fontsize=12)
axes[0,0].set_ylabel('% Remaining', fontsize=12)
axes[0,0].set_title('Data Remaining vs K', fontsize=14, fontweight='bold')
axes[0,0].legend(fontsize=11)
axes[0,0].grid(True, alpha=0.3)
axes[0,0].set_xticks(K_VALUES)

# Chart 2: % transactions + buyers survived — MOST IMPORTANT
axes[0,1].plot(res['K'], res['pct_trans'], 'o-', color='#e74c3c', label='% Transactions', linewidth=2)
axes[0,1].plot(res['K'], res['pct_buyers'], 's-', color='#2ecc71', label='% Buyers', linewidth=2)
axes[0,1].set_xlabel('K', fontsize=12)
axes[0,1].set_ylabel('% Survived', fontsize=12)
axes[0,1].set_title('Transactions & Buyers Survived vs K\n(THIS IS THE KEY CHART)', 
                     fontsize=14, fontweight='bold')
axes[0,1].legend(fontsize=11)
axes[0,1].grid(True, alpha=0.3)
axes[0,1].set_xticks(K_VALUES)

# Chart 3: Density
axes[1,0].plot(res['K'], res['density'], 'o-', color='coral', linewidth=2)
axes[1,0].set_xlabel('K', fontsize=12)
axes[1,0].set_ylabel('Density %', fontsize=12)
axes[1,0].set_title('Matrix Density vs K\n(higher = better for CF)', fontsize=14, fontweight='bold')
axes[1,0].grid(True, alpha=0.3)
axes[1,0].set_xticks(K_VALUES)

# Chart 4: Avg events per user
axes[1,1].plot(res['K'], res['avg_events_per_user'], 'o-', color='steelblue', linewidth=2)
axes[1,1].set_xlabel('K', fontsize=12)
axes[1,1].set_ylabel('Avg Events/User', fontsize=12)
axes[1,1].set_title('Avg Events per User vs K\n(higher = better sequences)', fontsize=14, fontweight='bold')
axes[1,1].grid(True, alpha=0.3)
axes[1,1].set_xticks(K_VALUES)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/kcore_simulation.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\n💾 {FIG_DIR}/kcore_simulation.png")

print(f"""
\nLook at:
  1. Chart top-right: % Transactions survived — drop below 50% là mất quá nhiều positive labels
  2. Chart bottom-left: Density — cần > 0.01% cho CF, > 0.05% tốt hơn
  3. Chart bottom-right: Avg events/user — cần > 5 cho meaningful sequences
  
Trade-off: K cao → density + sequence quality tốt hơn, nhưng mất transactions + buyers
""")

Baseline (after bot removal):
  Events:          2,714,354
  Users:           1,407,551
  Items:             234,151
  Transactions:       20,377
  Buyers:             11,697

  K-CORE SIMULATION

  K     Events   %Evts    Users   %Usrs    Items   %Itms   Trans  %Trans  Buyers   %Byrs   Density  Avg/Usr
-----------------------------------------------------------------------------------------------
  2  1,644,090   60.6%  387,060   27.5%  112,685   48.1%  20,304   99.6%  11,627   99.4%   0.0038%      4.2
  3  1,184,358   43.6%  179,623   12.8%   67,388   28.8%  19,669   96.5%  11,020   94.2%   0.0098%      6.6
  4    916,297   33.8%  102,356    7.3%   46,667   19.9%  17,492   85.8%   8,977   76.7%   0.0192%      9.0
  5    739,445   27.2%   65,559    4.7%   35,099   15.0%  15,557   76.3%   7,350   62.8%   0.0321%     11.3
  6    606,109   22.3%   44,349    3.2%   27,181   11.6%  13,807   67.8%   6,013   51.4%   0.0503%     13.7
  7    501,627   18.5%   31,182    2.2%   21,302    9.1%  1

C:\Users\NITRO V 15\AppData\Local\Temp\ipykernel_20844\563762613.py:164: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
K = 5
before = len(events)
before_users = events['visitorid'].nunique()
before_items = events['itemid'].nunique()
before_trans = (events['event'] == 'transaction').sum()
before_buyers = events[events['event'] == 'transaction']['visitorid'].nunique()
 
prev_len = len(events)
iteration = 0
while True:
    ic = events.groupby('itemid').size()
    events = events[events['itemid'].isin(ic[ic >= K].index)]
    uc = events.groupby('visitorid').size()
    events = events[events['visitorid'].isin(uc[uc >= K].index)]
    iteration += 1
    if len(events) == prev_len:
        break
    prev_len = len(events)
 
after = len(events)
after_users = events['visitorid'].nunique()
after_items = events['itemid'].nunique()
after_trans = (events['event'] == 'transaction').sum()
after_buyers = events[events['event'] == 'transaction']['visitorid'].nunique()
 
print(f"K-core K={K} applied ({iteration} iterations)")
print(f"  Events:       {before:>10,} → {after:>10,} ({after/before*100:.1f}%)")
print(f"  Users:        {before_users:>10,} → {after_users:>10,} ({after_users/before_users*100:.1f}%)")
print(f"  Items:        {before_items:>10,} → {after_items:>10,} ({after_items/before_items*100:.1f}%)")
print(f"  Transactions: {before_trans:>10,} → {after_trans:>10,} ({after_trans/before_trans*100:.1f}%)")
print(f"  Buyers:       {before_buyers:>10,} → {after_buyers:>10,} ({after_buyers/before_buyers*100:.1f}%)")
print(f"  Density:      {after/(after_users*after_items)*100:.4f}%")
print(f"\n✅ events df updated — now run session threshold analysis")

K-core K=5 applied (7 iterations)
  Events:        2,714,354 →    739,445 (27.2%)
  Users:         1,407,551 →     65,559 (4.7%)
  Items:           234,151 →     35,099 (15.0%)
  Transactions:     20,377 →     15,557 (76.3%)
  Buyers:           11,697 →      7,350 (62.8%)
  Density:      0.0321%

✅ events df updated — now run session threshold analysis


## 2D: Session threshold

In [12]:
"""
STEP 1D: SESSION THRESHOLD ANALYSIS
Assumes: events df đã bot-cleaned + K-core filtered trong memory.

Mục tiêu: tìm threshold (gap giữa 2 events) để define session boundary.
Nếu gap > threshold → session mới.

Phân tích:
  1. Distribution of inter-event gaps — tìm bimodal pattern
     (within-session gaps = short, between-session gaps = long)
  2. ECDF of gaps — tìm elbow/natural break point
  3. Thử nhiều thresholds (5, 10, 15, 20, 30, 45, 60 min)
     → show: sessions count, avg length, purchase sessions %

Output: charts + table → bạn chọn threshold
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

FIG_DIR = "outputs/figures"
os.makedirs(FIG_DIR, exist_ok=True)

if 'timestamp_dt' not in events.columns:
    events['timestamp_dt'] = pd.to_datetime(events['timestamp'], unit='ms')

# ============================================================
# 1. COMPUTE INTER-EVENT GAPS
# ============================================================
print("="*60)
print("  1. INTER-EVENT GAP DISTRIBUTION")
print("="*60)

events_sorted = events.sort_values(['visitorid', 'timestamp_dt']).copy()
events_sorted['prev_ts'] = events_sorted.groupby('visitorid')['timestamp_dt'].shift(1)
events_sorted['gap_sec'] = (events_sorted['timestamp_dt'] - events_sorted['prev_ts']).dt.total_seconds()

# Chỉ lấy gaps (bỏ first event per user = NaN gap)
gaps = events_sorted['gap_sec'].dropna()
gaps_minutes = gaps / 60

print(f"Total gaps: {len(gaps):,}")
print(f"\nGap distribution (minutes):")
for p, q in [('P10', 0.1), ('P25', 0.25), ('median', 0.5), ('P75', 0.75),
             ('P90', 0.9), ('P95', 0.95), ('P99', 0.99)]:
    val = gaps_minutes.quantile(q)
    print(f"  {p:8s}: {val:>10.1f} min ({val/60:.1f} hours)")

# ============================================================
# 2. CHARTS — Gap Distribution + ECDF
# ============================================================
print("\n" + "="*60)
print("  2. GAP DISTRIBUTION CHARTS")
print("="*60)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Chart 1: Histogram of gaps (0-120 minutes) — linear scale
axes[0,0].hist(gaps_minutes.clip(upper=120), bins=120, color='steelblue', edgecolor='white', alpha=0.8)
axes[0,0].set_xlabel('Gap (minutes)', fontsize=11)
axes[0,0].set_ylabel('Count', fontsize=11)
axes[0,0].set_title('Gap Distribution (0-120 min)', fontsize=13, fontweight='bold')
# Add vertical lines at candidate thresholds
for thresh, color in [(10, '#e74c3c'), (20, '#e67e22'), (30, '#2ecc71'), (60, '#3498db')]:
    axes[0,0].axvline(x=thresh, color=color, linestyle='--', alpha=0.7, label=f'{thresh}min')
axes[0,0].legend(fontsize=9)

# Chart 2: Histogram of gaps (log scale) — shows bimodal pattern
log_gaps = gaps_minutes[gaps_minutes > 0]
axes[0,1].hist(np.log10(log_gaps.clip(lower=0.01, upper=10000)), bins=100,
               color='coral', edgecolor='white', alpha=0.8)
axes[0,1].set_xlabel('log10(Gap in minutes)', fontsize=11)
axes[0,1].set_ylabel('Count', fontsize=11)
axes[0,1].set_title('Gap Distribution (log scale)\nBimodal = natural session boundary', 
                     fontsize=13, fontweight='bold')
# Mark candidate thresholds
for thresh, color in [(10, '#e74c3c'), (20, '#e67e22'), (30, '#2ecc71'), (60, '#3498db')]:
    axes[0,1].axvline(x=np.log10(thresh), color=color, linestyle='--', alpha=0.7, label=f'{thresh}min')
axes[0,1].legend(fontsize=9)

# Chart 3: ECDF of gaps (0-180 min zoom)
sorted_gaps = np.sort(gaps_minutes.values)
ecdf_y = np.arange(1, len(sorted_gaps) + 1) / len(sorted_gaps)
# Subsample for plotting (too many points)
step = max(1, len(sorted_gaps) // 50000)
axes[1,0].plot(sorted_gaps[::step], ecdf_y[::step], linewidth=1.5, color='steelblue')
axes[1,0].set_xlabel('Gap (minutes)', fontsize=11)
axes[1,0].set_ylabel('Cumulative Proportion', fontsize=11)
axes[1,0].set_title('ECDF of Gaps (full range)', fontsize=13, fontweight='bold')
axes[1,0].set_xscale('log')
for thresh, color in [(10, '#e74c3c'), (20, '#e67e22'), (30, '#2ecc71'), (60, '#3498db')]:
    axes[1,0].axvline(x=thresh, color=color, linestyle='--', alpha=0.7, label=f'{thresh}min')
axes[1,0].legend(fontsize=9)
axes[1,0].grid(True, alpha=0.3)

# Chart 4: ECDF zoomed 0-120 min
mask_zoom = sorted_gaps <= 120
if mask_zoom.sum() > 0:
    sg_zoom = sorted_gaps[mask_zoom]
    ec_zoom = ecdf_y[mask_zoom]
    step2 = max(1, len(sg_zoom) // 10000)
    axes[1,1].plot(sg_zoom[::step2], ec_zoom[::step2], linewidth=1.5, color='steelblue')
    axes[1,1].set_xlabel('Gap (minutes)', fontsize=11)
    axes[1,1].set_ylabel('Cumulative Proportion', fontsize=11)
    axes[1,1].set_title('ECDF Zoomed (0-120 min) — find elbow', fontsize=13, fontweight='bold')
    for thresh, color in [(10, '#e74c3c'), (20, '#e67e22'), (30, '#2ecc71'), (60, '#3498db')]:
        axes[1,1].axvline(x=thresh, color=color, linestyle='--', alpha=0.7, label=f'{thresh}min')
    axes[1,1].legend(fontsize=9)
    axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/session_threshold_gaps.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"💾 {FIG_DIR}/session_threshold_gaps.png")

# % gaps below each threshold
print(f"\n% of gaps below each threshold:")
for t in [5, 10, 15, 20, 30, 45, 60, 90, 120]:
    pct_below = (gaps_minutes <= t).mean() * 100
    print(f"  ≤{t:>3d} min: {pct_below:.1f}% of gaps")


# ============================================================
# 3. SESSION METRICS AT DIFFERENT THRESHOLDS
# ============================================================
print("\n" + "="*60)
print("  3. SESSION METRICS BY THRESHOLD")
print("="*60)

THRESHOLD_CANDIDATES = [5, 10, 15, 20, 30, 45, 60]

results = []
for thresh_min in THRESHOLD_CANDIDATES:
    thresh_sec = thresh_min * 60
    
    # Define sessions
    temp = events_sorted.copy()
    temp['new_session'] = (temp['gap_sec'] > thresh_sec) | temp['gap_sec'].isna()
    temp['session_id'] = temp.groupby('visitorid')['new_session'].cumsum()
    
    # Session stats
    sess = temp.groupby(['visitorid', 'session_id']).agg(
        n_events=('event', 'count'),
        n_items=('itemid', 'nunique'),
        has_trans=('event', lambda x: (x == 'transaction').any()),
        has_cart=('event', lambda x: (x == 'addtocart').any()),
        duration_sec=('timestamp_dt', lambda x: (x.max() - x.min()).total_seconds()),
    )
    
    n_sessions = len(sess)
    n_purchase_sessions = sess['has_trans'].sum()
    active_sess = sess[sess['duration_sec'] > 0]
    
    results.append({
        'threshold_min': thresh_min,
        'n_sessions': n_sessions,
        'avg_events_per_session': sess['n_events'].mean(),
        'median_events': sess['n_events'].median(),
        'avg_items_per_session': sess['n_items'].mean(),
        'pct_purchase_sessions': n_purchase_sessions / n_sessions * 100,
        'pct_cart_sessions': sess['has_cart'].sum() / n_sessions * 100,
        'median_duration_min': active_sess['duration_sec'].median() / 60 if len(active_sess) > 0 else 0,
        'sessions_per_user': n_sessions / events['visitorid'].nunique(),
    })

res = pd.DataFrame(results)

print(f"\n{'Thresh':>7s} {'Sessions':>10s} {'Avg Evts':>9s} {'Med Evts':>9s} {'Avg Items':>10s} "
      f"{'%Purchase':>10s} {'%Cart':>7s} {'Med Dur':>8s} {'Sess/Usr':>9s}")
print(f"{'-'*82}")
for _, r in res.iterrows():
    print(f"{r['threshold_min']:>5.0f}m {r['n_sessions']:>10,.0f} {r['avg_events_per_session']:>9.1f} "
          f"{r['median_events']:>9.0f} {r['avg_items_per_session']:>10.1f} "
          f"{r['pct_purchase_sessions']:>9.2f}% {r['pct_cart_sessions']:>6.2f}% "
          f"{r['median_duration_min']:>7.1f}m {r['sessions_per_user']:>9.2f}")

# Chart
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(res['threshold_min'], res['n_sessions'], 'o-', color='steelblue', linewidth=2)
axes[0].set_xlabel('Threshold (minutes)', fontsize=11)
axes[0].set_ylabel('Number of Sessions', fontsize=11)
axes[0].set_title('Sessions vs Threshold', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)

axes[1].plot(res['threshold_min'], res['avg_events_per_session'], 'o-', color='coral', linewidth=2)
axes[1].set_xlabel('Threshold (minutes)', fontsize=11)
axes[1].set_ylabel('Avg Events per Session', fontsize=11)
axes[1].set_title('Session Richness vs Threshold', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)

axes[2].plot(res['threshold_min'], res['pct_purchase_sessions'], 'o-', color='#2ecc71', linewidth=2)
axes[2].set_xlabel('Threshold (minutes)', fontsize=11)
axes[2].set_ylabel('% Sessions with Purchase', fontsize=11)
axes[2].set_title('Purchase Sessions vs Threshold', fontsize=13, fontweight='bold')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/session_threshold_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\n💾 {FIG_DIR}/session_threshold_metrics.png")

print(f"""
\nCách chọn:
  - Threshold quá nhỏ (5min) → sessions bị cắt vụn, mỗi session 1-2 events → sequences quá ngắn
  - Threshold quá lớn (60min) → merge nhiều visits khác nhau thành 1 session → noisy
  - Sweet spot: threshold nơi avg events/session đủ lớn (>3) mà sessions vẫn coherent
  - Nhìn log histogram: nếu có valley (bimodal) → threshold ở valley = natural boundary
""")

# Cleanup
del events_sorted

  1. INTER-EVENT GAP DISTRIBUTION
Total gaps: 673,886

Gap distribution (minutes):
  P10     :        0.2 min (0.0 hours)
  P25     :        0.6 min (0.0 hours)
  median  :        2.0 min (0.0 hours)
  P75     :       20.1 min (0.3 hours)
  P90     :     1432.2 min (23.9 hours)
  P95     :     6188.5 min (103.1 hours)
  P99     :    45181.7 min (753.0 hours)

  2. GAP DISTRIBUTION CHARTS


C:\Users\NITRO V 15\AppData\Local\Temp\ipykernel_20844\742923912.py:115: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


💾 outputs/figures/session_threshold_gaps.png

% of gaps below each threshold:
  ≤  5 min: 64.0% of gaps
  ≤ 10 min: 70.6% of gaps
  ≤ 15 min: 73.4% of gaps
  ≤ 20 min: 75.0% of gaps
  ≤ 30 min: 76.9% of gaps
  ≤ 45 min: 78.5% of gaps
  ≤ 60 min: 79.5% of gaps
  ≤ 90 min: 80.8% of gaps
  ≤120 min: 81.7% of gaps

  3. SESSION METRICS BY THRESHOLD

 Thresh   Sessions  Avg Evts  Med Evts  Avg Items  %Purchase   %Cart  Med Dur  Sess/Usr
----------------------------------------------------------------------------------
    5m    308,235       2.4         1        1.8      3.57%   8.63%     2.9m      4.70
   10m    263,430       2.8         1        2.1      3.98%   9.33%     4.4m      4.02
   15m    245,022       3.0         2        2.2      4.13%   9.63%     5.4m      3.74
   20m    234,189       3.2         2        2.2      4.19%   9.78%     6.1m      3.57
   30m    221,345       3.3         2        2.3      4.25%   9.98%     7.1m      3.38
   45m    210,491       3.5         2        2

C:\Users\NITRO V 15\AppData\Local\Temp\ipykernel_20844\742923912.py:202: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## EDA for Feature Engineering 

In [13]:
"""
STEP 1E: APPLY SESSION (30min) + FULL EDA ON FINAL CLEAN DATA
Assumes: events df đã K-core filtered (K=5) trong memory.

Output:
  - Session columns gắn vào events
  - Full EDA: mọi thứ cần cho feature engineering + embedding decisions
  - Figures saved
  - events FINAL saved
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import json
import os

FIG_DIR = "outputs/figures"
OUT_DIR = "data/processed"
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

stats = {}

def section(title):
    print(f"\n{'='*70}")
    print(f"  {title}")
    print(f"{'='*70}")


# ==============================================================
# PART 0: APPLY SESSION THRESHOLD = 30 MIN
# ==============================================================
section("0. APPLY SESSION THRESHOLD (30 min)")

SESSION_THRESHOLD_MIN = 20

if 'timestamp_dt' not in events.columns:
    events['timestamp_dt'] = pd.to_datetime(events['timestamp'], unit='ms')

events = events.sort_values(['visitorid', 'timestamp_dt']).reset_index(drop=True)

events['_prev_ts'] = events.groupby('visitorid')['timestamp_dt'].shift(1)
events['_gap_sec'] = (events['timestamp_dt'] - events['_prev_ts']).dt.total_seconds()

threshold_sec = SESSION_THRESHOLD_MIN * 60
events['_new_session'] = (events['_gap_sec'] > threshold_sec) | events['_gap_sec'].isna()
events['session_id'] = events.groupby('visitorid')['_new_session'].cumsum()

# Session columns
events['session_position'] = events.groupby(['visitorid', 'session_id']).cumcount()

session_length = events.groupby(['visitorid', 'session_id']).size().reset_index(name='session_length')
events = events.merge(session_length, on=['visitorid', 'session_id'], how='left')

events['session_unique_items'] = events.groupby(['visitorid', 'session_id'])['itemid'].transform('nunique')

events['session_has_purchase'] = events.groupby(['visitorid', 'session_id'])['event'].transform(
    lambda x: (x == 'transaction').any().astype(int)
)
events['session_has_cart'] = events.groupby(['visitorid', 'session_id'])['event'].transform(
    lambda x: (x == 'addtocart').any().astype(int)
)

session_start = events.groupby(['visitorid', 'session_id'])['timestamp_dt'].transform('min')
events['session_duration_sec'] = (
    events.groupby(['visitorid', 'session_id'])['timestamp_dt'].transform('max') - session_start
).dt.total_seconds()
events['time_in_session_sec'] = (events['timestamp_dt'] - session_start).dt.total_seconds()

# Time features
events['hour'] = events['timestamp_dt'].dt.hour
events['day_of_week'] = events['timestamp_dt'].dt.dayofweek
events['date'] = events['timestamp_dt'].dt.date

events = events.drop(columns=['_prev_ts', '_gap_sec', '_new_session'])

n_sessions = events.groupby(['visitorid', 'session_id']).ngroups
print(f"Session threshold: {SESSION_THRESHOLD_MIN} min")
print(f"Sessions: {n_sessions:,}")
print(f"Avg events/session: {len(events)/n_sessions:.1f}")
print(f"Columns: {list(events.columns)}")


# ==============================================================
# PART 1: BASIC STATS
# ==============================================================
section("1. BASIC STATS — FINAL CLEAN DATA")

N = len(events)
N_USERS = events['visitorid'].nunique()
N_ITEMS = events['itemid'].nunique()
DENSITY = N / (N_USERS * N_ITEMS) * 100

n_views = (events['event'] == 'view').sum()
n_carts = (events['event'] == 'addtocart').sum()
n_trans = (events['event'] == 'transaction').sum()
n_buyers = events[events['event'] == 'transaction']['visitorid'].nunique()

stats.update({
    'n_events': int(N), 'n_users': int(N_USERS), 'n_items': int(N_ITEMS),
    'density_pct': round(DENSITY, 4),
    'n_views': int(n_views), 'n_carts': int(n_carts), 'n_trans': int(n_trans),
    'n_buyers': int(n_buyers), 'n_sessions': int(n_sessions),
})

print(f"""
  Events:       {N:>10,}
  Users:        {N_USERS:>10,}
  Items:        {N_ITEMS:>10,}
  Density:      {DENSITY:>10.4f}%
  
  Views:        {n_views:>10,} ({n_views/N*100:.1f}%)
  AddToCart:    {n_carts:>10,} ({n_carts/N*100:.1f}%)
  Transactions: {n_trans:>10,} ({n_trans/N*100:.1f}%)
  Buyers:       {n_buyers:>10,} ({n_buyers/N_USERS*100:.1f}% of users)
  Sessions:     {n_sessions:>10,}
""")


# ==============================================================
# PART 2: CONVERSION FUNNEL
# ==============================================================
section("2. CONVERSION FUNNEL")

view_to_cart = n_carts / max(n_views, 1) * 100
cart_to_trans = n_trans / max(n_carts, 1) * 100
view_to_trans = n_trans / max(n_views, 1) * 100

print(f"  View → Cart:        {view_to_cart:.2f}%")
print(f"  Cart → Transaction: {cart_to_trans:.2f}%")
print(f"  View → Transaction: {view_to_trans:.2f}%")

stats.update({
    'funnel_view_cart': round(view_to_cart, 2),
    'funnel_cart_trans': round(cart_to_trans, 2),
    'funnel_view_trans': round(view_to_trans, 2),
})


# ==============================================================
# PART 3: USER-ITEM PAIR OUTCOMES (Label Design Input)
# ==============================================================
section("3. USER-ITEM PAIR OUTCOMES")

user_item = events.groupby(['visitorid', 'itemid']).agg(
    n_views=('event', lambda x: (x == 'view').sum()),
    n_carts=('event', lambda x: (x == 'addtocart').sum()),
    n_trans=('event', lambda x: (x == 'transaction').sum()),
    total_events=('event', 'count'),
    first_ts=('timestamp_dt', 'min'),
    last_ts=('timestamp_dt', 'max'),
    n_sessions_involved=('session_id', 'nunique'),
).reset_index()

purchased = (user_item['n_trans'] > 0).sum()
carted_only = ((user_item['n_carts'] > 0) & (user_item['n_trans'] == 0)).sum()
viewed_only = ((user_item['n_carts'] == 0) & (user_item['n_trans'] == 0)).sum()
total_pairs = len(user_item)

stats.update({
    'n_pairs': int(total_pairs),
    'n_pairs_purchased': int(purchased),
    'n_pairs_carted': int(carted_only),
    'n_pairs_viewed': int(viewed_only),
})

print(f"  Total user-item pairs: {total_pairs:,}")
print(f"  Purchased:       {purchased:>10,} ({purchased/total_pairs*100:.2f}%)")
print(f"  Carted only:     {carted_only:>10,} ({carted_only/total_pairs*100:.2f}%)")
print(f"  Viewed only:     {viewed_only:>10,} ({viewed_only/total_pairs*100:.2f}%)")

# Repeat views before action
purchased_pairs = user_item[user_item['n_trans'] > 0]
carted_pairs = user_item[(user_item['n_carts'] > 0) & (user_item['n_trans'] == 0)]
viewed_pairs = user_item[(user_item['n_carts'] == 0) & (user_item['n_trans'] == 0)]

print(f"\n  Avg views before purchase: {purchased_pairs['n_views'].mean():.1f}")
print(f"  Avg views for carted-only: {carted_pairs['n_views'].mean():.1f}")
print(f"  Avg views for view-only:   {viewed_pairs['n_views'].mean():.1f}")
print(f"  → Repeat view count có thể là strong feature cho ranking")

print(f"\n  Sessions involved per pair:")
print(f"    Purchased pairs: avg {purchased_pairs['n_sessions_involved'].mean():.1f} sessions")
print(f"    Carted pairs:    avg {carted_pairs['n_sessions_involved'].mean():.1f} sessions")
print(f"    Viewed pairs:    avg {viewed_pairs['n_sessions_involved'].mean():.1f} sessions")
print(f"  → Multi-session interest = stronger signal")


# ==============================================================
# PART 4: USER ANALYSIS
# ==============================================================
section("4. USER ANALYSIS")

user_stats_df = events.groupby('visitorid').agg(
    total_events=('event', 'count'),
    n_views=('event', lambda x: (x == 'view').sum()),
    n_carts=('event', lambda x: (x == 'addtocart').sum()),
    n_trans=('event', lambda x: (x == 'transaction').sum()),
    unique_items=('itemid', 'nunique'),
    n_sessions=('session_id', 'nunique'),
    first_event=('timestamp_dt', 'min'),
    last_event=('timestamp_dt', 'max'),
).reset_index()

user_stats_df['has_purchase'] = user_stats_df['n_trans'] > 0
user_stats_df['lifespan_days'] = (user_stats_df['last_event'] - user_stats_df['first_event']).dt.days
user_stats_df['purchase_rate'] = user_stats_df['n_trans'] / user_stats_df['total_events']
user_stats_df['events_per_session'] = user_stats_df['total_events'] / user_stats_df['n_sessions']

print("User activity percentiles:")
for p, q in [('P25', 0.25), ('P50', 0.5), ('P75', 0.75), ('P90', 0.9), ('P95', 0.95)]:
    val = user_stats_df['total_events'].quantile(q)
    print(f"  {p}: {val:.0f} events")

# User segments
n_browsers = (~user_stats_df['has_purchase'] & (user_stats_df['n_carts'] == 0)).sum()
n_carters = (~user_stats_df['has_purchase'] & (user_stats_df['n_carts'] > 0)).sum()

print(f"\nUser types:")
print(f"  Browsers (view only): {n_browsers:>8,} ({n_browsers/N_USERS*100:.1f}%)")
print(f"  Carters (cart, no buy):{n_carters:>8,} ({n_carters/N_USERS*100:.1f}%)")
print(f"  Buyers:               {n_buyers:>8,} ({n_buyers/N_USERS*100:.1f}%)")

# Returning vs one-time
returning = (user_stats_df['lifespan_days'] > 0).sum()
one_time = (user_stats_df['lifespan_days'] == 0).sum()
print(f"\n  Single-day users: {one_time:>8,} ({one_time/N_USERS*100:.1f}%)")
print(f"  Returning users:  {returning:>8,} ({returning/N_USERS*100:.1f}%)")

if returning > 0:
    ret = user_stats_df[user_stats_df['lifespan_days'] > 0]
    print(f"  Returning lifespan: median {ret['lifespan_days'].median():.0f} days")

stats.update({
    'pct_browsers': round(n_browsers/N_USERS*100, 1),
    'pct_carters': round(n_carters/N_USERS*100, 1),
    'pct_buyers': round(n_buyers/N_USERS*100, 1),
    'pct_returning': round(returning/N_USERS*100, 1),
    'median_events_per_user': float(user_stats_df['total_events'].median()),
    'median_sessions_per_user': float(user_stats_df['n_sessions'].median()),
})

# --- Plot ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(user_stats_df['total_events'].clip(upper=50), bins=50,
             color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Events per User')
axes[0].set_title('User Activity Distribution', fontweight='bold')

axes[1].hist(user_stats_df['n_sessions'].clip(upper=20), bins=20,
             color='coral', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('Sessions per User')
axes[1].set_title('Sessions per User', fontweight='bold')

# Buyer rate by activity quantile
user_stats_df['activity_q'] = pd.qcut(user_stats_df['total_events'], q=5, labels=False, duplicates='drop')
n_bins = user_stats_df['activity_q'].nunique()
bin_labels = [f'Q{i+1}' for i in range(n_bins)]
user_stats_df['activity_q'] = user_stats_df['activity_q'].map(dict(enumerate(bin_labels)))
buyer_by_q = user_stats_df.groupby('activity_q', observed=True)['has_purchase'].mean() * 100
axes[2].bar(range(len(buyer_by_q)), buyer_by_q.values, color='#2ecc71', edgecolor='white')
axes[2].set_xticks(range(len(buyer_by_q)))
axes[2].set_xticklabels(buyer_by_q.index)
axes[2].set_ylabel('% Buyers')
axes[2].set_title('Purchase Rate by Activity Quintile', fontweight='bold')

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/eda_01_users.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"💾 {FIG_DIR}/eda_01_users.png")


# ==============================================================
# PART 5: ITEM ANALYSIS
# ==============================================================
section("5. ITEM ANALYSIS")

item_stats_df = events.groupby('itemid').agg(
    total_events=('event', 'count'),
    n_views=('event', lambda x: (x == 'view').sum()),
    n_carts=('event', lambda x: (x == 'addtocart').sum()),
    n_trans=('event', lambda x: (x == 'transaction').sum()),
    unique_visitors=('visitorid', 'nunique'),
).reset_index()

item_stats_df['conversion_rate'] = item_stats_df['n_trans'] / item_stats_df['n_views'].clip(lower=1)
item_stats_df['cart_rate'] = item_stats_df['n_carts'] / item_stats_df['n_views'].clip(lower=1)
item_stats_df['has_transaction'] = item_stats_df['n_trans'] > 0

items_with_trans = item_stats_df['has_transaction'].sum()

print(f"  Items with ≥1 transaction: {items_with_trans:,} ({items_with_trans/N_ITEMS*100:.1f}%)")
print(f"  Items view-only: {N_ITEMS - items_with_trans:,}")

# Long tail
sorted_pop = item_stats_df['total_events'].sort_values(ascending=False)
cumsum = sorted_pop.cumsum()
total = sorted_pop.sum()

print(f"\n  Long tail:")
for p in [1, 5, 10, 20]:
    n_top = max(1, int(len(sorted_pop) * p / 100))
    pct_int = cumsum.iloc[n_top-1] / total * 100
    print(f"    Top {p:2d}% items → {pct_int:.1f}% events")

# Gini
n_arr = np.arange(1, len(sorted_pop)+1) / len(sorted_pop)
c_arr = cumsum.values / total
gini = 1 - 2 * np.trapz(c_arr, n_arr)
print(f"  Gini: {gini:.4f}")
stats['gini_item'] = round(gini, 4)

# Conversion rate for items with enough views
items_10v = item_stats_df[item_stats_df['n_views'] >= 10]
if len(items_10v) > 0:
    print(f"\n  Conversion rate (items ≥10 views, n={len(items_10v):,}):")
    print(f"    Mean:   {items_10v['conversion_rate'].mean():.4f}")
    print(f"    Median: {items_10v['conversion_rate'].median():.4f}")
    print(f"    P90:    {items_10v['conversion_rate'].quantile(0.9):.4f}")

# --- Plot ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ip_counts = item_stats_df['total_events'].value_counts().sort_index()
axes[0].scatter(ip_counts.index, ip_counts.values, s=3, alpha=0.5, color='steelblue')
axes[0].set_xscale('log')
axes[0].set_yscale('log')
axes[0].set_xlabel('Events per Item')
axes[0].set_title('Item Popularity (log-log)', fontweight='bold')

axes[1].plot(n_arr, c_arr, color='coral', linewidth=2)
axes[1].plot([0,1],[0,1], 'k--', alpha=0.3)
axes[1].set_xlabel('Proportion of Items')
axes[1].set_ylabel('Proportion of Events')
axes[1].set_title(f'Lorenz Curve (Gini={gini:.3f})', fontweight='bold')

if len(items_10v) > 0:
    axes[2].hist(items_10v['conversion_rate'].clip(upper=0.2), bins=50,
                 color='#2ecc71', edgecolor='white', alpha=0.8)
    axes[2].set_xlabel('Conversion Rate')
    axes[2].set_title('Item Conversion Rate (≥10 views)', fontweight='bold')

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/eda_02_items.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"💾 {FIG_DIR}/eda_02_items.png")


# ==============================================================
# PART 6: SESSION ANALYSIS
# ==============================================================
section("6. SESSION ANALYSIS")

sess_df = events.groupby(['visitorid', 'session_id']).agg(
    n_events=('event', 'count'),
    n_items=('itemid', 'nunique'),
    has_purchase=('session_has_purchase', 'first'),
    has_cart=('session_has_cart', 'first'),
    duration_sec=('session_duration_sec', 'first'),
).reset_index()

print(f"  Total sessions: {len(sess_df):,}")
print(f"  Purchase sessions: {sess_df['has_purchase'].sum():,} ({sess_df['has_purchase'].mean()*100:.2f}%)")
print(f"  Cart sessions: {sess_df['has_cart'].sum():,} ({sess_df['has_cart'].mean()*100:.2f}%)")

print(f"\n  Session length (events):")
for p, q in [('P25', 0.25), ('P50', 0.5), ('P75', 0.75), ('P90', 0.9)]:
    print(f"    {p}: {sess_df['n_events'].quantile(q):.0f}")

print(f"\n  Session items:")
for p, q in [('P50', 0.5), ('P75', 0.75), ('P90', 0.9)]:
    print(f"    {p}: {sess_df['n_items'].quantile(q):.0f}")

active = sess_df[sess_df['duration_sec'] > 0]
if len(active) > 0:
    print(f"\n  Session duration (active sessions, n={len(active):,}):")
    for p, q in [('P50', 0.5), ('P75', 0.75), ('P90', 0.9)]:
        val = active['duration_sec'].quantile(q)
        print(f"    {p}: {val:.0f}s ({val/60:.1f}min)")

# Purchase sessions vs non-purchase sessions
purchase_sess = sess_df[sess_df['has_purchase'] == 1]
no_purchase_sess = sess_df[sess_df['has_purchase'] == 0]

print(f"\n  Purchase sessions avg events: {purchase_sess['n_events'].mean():.1f}")
print(f"  Non-purchase sessions avg events: {no_purchase_sess['n_events'].mean():.1f}")
print(f"  → Purchase sessions are longer → session_length is a feature")

stats.update({
    'median_session_events': float(sess_df['n_events'].median()),
    'pct_purchase_sessions': round(sess_df['has_purchase'].mean()*100, 2),
    'avg_events_purchase_session': round(purchase_sess['n_events'].mean(), 1),
    'avg_events_nonpurchase_session': round(no_purchase_sess['n_events'].mean(), 1),
})

# --- Plot ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(sess_df['n_events'].clip(upper=20), bins=20, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Events per Session')
axes[0].set_title('Session Length', fontweight='bold')

if len(active) > 0:
    axes[1].hist((active['duration_sec']/60).clip(upper=60), bins=50, color='coral', edgecolor='white', alpha=0.8)
    axes[1].set_xlabel('Duration (minutes)')
    axes[1].set_title('Session Duration', fontweight='bold')

# Purchase rate by session length
sess_df['length_bin'] = pd.cut(sess_df['n_events'], bins=[0,1,2,3,5,10,20,1000],
                                labels=['1','2','3','4-5','6-10','11-20','20+'])
pr_by_len = sess_df.groupby('length_bin', observed=True)['has_purchase'].mean() * 100
axes[2].bar(range(len(pr_by_len)), pr_by_len.values, color='#2ecc71', edgecolor='white')
axes[2].set_xticks(range(len(pr_by_len)))
axes[2].set_xticklabels(pr_by_len.index, fontsize=9)
axes[2].set_xlabel('Session Length (events)')
axes[2].set_ylabel('% Purchase')
axes[2].set_title('Purchase Rate by Session Length', fontweight='bold')

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/eda_03_sessions.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"💾 {FIG_DIR}/eda_03_sessions.png")


# ==============================================================
# PART 7: TEMPORAL ANALYSIS
# ==============================================================
section("7. TEMPORAL ANALYSIS")

daily = events.groupby('date').agg(
    total=('event', 'count'),
    trans=('event', lambda x: (x == 'transaction').sum()),
).reset_index()
daily['conv_rate'] = daily['trans'] / daily['total'] * 100

hourly = events.groupby('hour').size()
dow = events.groupby('day_of_week').size()
dow_names = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']

print(f"  Daily events: mean={daily['total'].mean():.0f}, median={daily['total'].median():.0f}")
print(f"  Daily conversion rate: mean={daily['conv_rate'].mean():.3f}%")

print(f"\n  Peak hours: {hourly.nlargest(3).index.tolist()}")
print(f"  Peak days: {[dow_names[d] for d in dow.nlargest(3).index.tolist()]}")

# Conversion by hour
hourly_trans = events[events['event'] == 'transaction'].groupby('hour').size()
hourly_views = events[events['event'] == 'view'].groupby('hour').size()
hourly_conv = (hourly_trans / hourly_views * 100).fillna(0)

print(f"\n  Conversion rate by hour (top 5):")
for h in hourly_conv.nlargest(5).index:
    print(f"    {h:02d}:00 → {hourly_conv[h]:.2f}%")

# --- Plot ---
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0,0].plot(range(len(daily)), daily['total'].values, linewidth=0.8, color='steelblue')
axes[0,0].set_title('Daily Events', fontweight='bold')
tick_step = max(1, len(daily)//10)
axes[0,0].set_xticks(range(0, len(daily), tick_step))
axes[0,0].set_xticklabels([str(daily['date'].iloc[i]) for i in range(0, len(daily), tick_step)],
                           rotation=45, fontsize=8)

axes[0,1].plot(range(len(daily)), daily['conv_rate'].values, linewidth=0.8, color='coral')
axes[0,1].set_title('Daily Conversion Rate %', fontweight='bold')
axes[0,1].set_xticks(range(0, len(daily), tick_step))
axes[0,1].set_xticklabels([str(daily['date'].iloc[i]) for i in range(0, len(daily), tick_step)],
                           rotation=45, fontsize=8)

axes[1,0].bar(range(24), [hourly.get(h,0) for h in range(24)], color='steelblue', edgecolor='white')
axes[1,0].set_xlabel('Hour')
axes[1,0].set_title('Hourly Pattern', fontweight='bold')

axes[1,1].bar(range(7), [dow.get(d,0) for d in range(7)], color='coral', edgecolor='white')
axes[1,1].set_xticks(range(7))
axes[1,1].set_xticklabels(dow_names)
axes[1,1].set_title('Day of Week', fontweight='bold')

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/eda_04_temporal.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"💾 {FIG_DIR}/eda_04_temporal.png")


# ==============================================================
# PART 8: TEMPORAL SPLIT SIMULATION
# ==============================================================
section("8. TEMPORAL SPLIT (70/15/15)")

train_cut = events['timestamp_dt'].quantile(0.70)
val_cut = events['timestamp_dt'].quantile(0.85)

print(f"  Train cutoff: {train_cut.date()}")
print(f"  Val cutoff:   {val_cut.date()}")

mask_train = events['timestamp_dt'] < train_cut
mask_val = (events['timestamp_dt'] >= train_cut) & (events['timestamp_dt'] < val_cut)
mask_test = events['timestamp_dt'] >= val_cut

for name, mask in [('Train', mask_train), ('Val', mask_val), ('Test', mask_test)]:
    n = mask.sum()
    n_t = ((events['event'] == 'transaction') & mask).sum()
    print(f"  {name:5s}: {n:>10,} events ({n/N*100:.1f}%), {n_t:>6,} transactions")

# Warm/cold
train_users = set(events.loc[mask_train, 'visitorid'].unique())
train_items = set(events.loc[mask_train, 'itemid'].unique())

for name, mask in [('Val', mask_val), ('Test', mask_test)]:
    s_users = set(events.loc[mask, 'visitorid'].unique())
    s_items = set(events.loc[mask, 'itemid'].unique())
    warm_u = len(s_users & train_users)
    cold_u = len(s_users - train_users)
    warm_i = len(s_items & train_items)
    cold_i = len(s_items - train_items)
    
    n_split = mask.sum()
    ww = events[mask & events['visitorid'].isin(train_users) & events['itemid'].isin(train_items)].shape[0]
    
    print(f"\n  {name}: {len(s_users):,} users (warm:{warm_u:,} cold:{cold_u:,}), "
          f"{len(s_items):,} items (warm:{warm_i:,} cold:{cold_i:,})")
    print(f"    Warm-warm events: {ww:,} ({ww/max(n_split,1)*100:.1f}%)")

stats['train_cutoff'] = str(train_cut.date())
stats['val_cutoff'] = str(val_cut.date())


# ==============================================================
# PART 9: ITEM CATEGORY COVERAGE
# ==============================================================
section("9. ITEM CATEGORY COVERAGE")

try:
    props = pd.concat([
        pd.read_csv("data/raw/item_properties_part1.csv", dtype=str),
        pd.read_csv("data/raw/item_properties_part2.csv", dtype=str),
    ])
    
    cat_props = props[props['property'] == 'categoryid']
    cat_items = set(cat_props['itemid'].unique())
    our_items = set(events['itemid'].unique())
    overlap = len(cat_items & our_items)
    
    print(f"  Items with categoryid: {overlap:,} / {N_ITEMS:,} ({overlap/N_ITEMS*100:.1f}%)")
    
    # Available property
    avail_props = props[props['property'] == 'available']
    avail_items = set(avail_props['itemid'].unique())
    avail_overlap = len(avail_items & our_items)
    print(f"  Items with 'available' property: {avail_overlap:,} / {N_ITEMS:,} ({avail_overlap/N_ITEMS*100:.1f}%)")
    
    stats['pct_items_with_category'] = round(overlap/N_ITEMS*100, 1)
    
    del props  # free memory
except Exception as e:
    print(f"  Could not load item properties: {e}")


# ==============================================================
# PART 10: EMBEDDING SIGNAL SUMMARY
# ==============================================================
section("10. SIGNALS AVAILABLE FOR EMBEDDINGS")

# View sequences per user (for Item2Vec)
view_events = events[events['event'] == 'view']
user_view_seqs = view_events.groupby('visitorid')['itemid'].apply(list)
seq_lengths = user_view_seqs.apply(len)
users_with_2plus = (seq_lengths >= 2).sum()

print(f"  Item2Vec (view sequences):")
print(f"    Users with ≥2 views: {users_with_2plus:,}")
print(f"    Sequence length: median={seq_lengths.median():.0f}, P75={seq_lengths.quantile(0.75):.0f}, P90={seq_lengths.quantile(0.9):.0f}")

# Session-level sequences (for session-based Item2Vec)
session_seqs = view_events.groupby(['visitorid', 'session_id'])['itemid'].apply(list)
session_seq_lens = session_seqs.apply(len)
sessions_2plus = (session_seq_lens >= 2).sum()

print(f"\n  Session-level Item2Vec:")
print(f"    Sessions with ≥2 views: {sessions_2plus:,}")
print(f"    Sequence length: median={session_seq_lens.median():.0f}, P75={session_seq_lens.quantile(0.75):.0f}")

# Graph edges (for LightGCN)
print(f"\n  LightGCN (user-item graph):")
print(f"    Edges: {N:,}")
print(f"    Edge types: view({n_views:,}), cart({n_carts:,}), trans({n_trans:,})")
print(f"    → Multi-weight: view=0.1, cart=0.5, trans=1.0")

# Full sequences (for SASRec)
full_seqs = events.groupby('visitorid')['itemid'].apply(list)
full_lens = full_seqs.apply(len)

print(f"\n  SASRec (full event sequences):")
print(f"    Users with ≥3 events: {(full_lens >= 3).sum():,}")
print(f"    Sequence length: median={full_lens.median():.0f}, P75={full_lens.quantile(0.75):.0f}, P90={full_lens.quantile(0.9):.0f}")
print(f"    → Max sequence length for model: recommend {int(full_lens.quantile(0.9))}")

stats.update({
    'item2vec_users': int(users_with_2plus),
    'item2vec_median_seq': float(seq_lengths.median()),
    'sasrec_users': int((full_lens >= 3).sum()),
    'sasrec_recommended_maxlen': int(full_lens.quantile(0.9)),
    'lightgcn_edges': int(N),
})


# ==============================================================
# SAVE
# ==============================================================
section("SAVE")

# Save final clean events
save_cols = ['timestamp', 'visitorid', 'event', 'itemid', 'transactionid',
             'session_id', 'session_position', 'session_length', 'session_unique_items',
             'session_has_purchase', 'session_has_cart', 'session_duration_sec',
             'time_in_session_sec', 'hour', 'day_of_week']
save_cols = [c for c in save_cols if c in events.columns]

events[save_cols].to_parquet(os.path.join(OUT_DIR, "events_final.parquet"), index=False)
print(f"✅ events_final.parquet: {len(events):,} rows × {len(save_cols)} cols")

# Save stats
with open(os.path.join(OUT_DIR, "eda_final_stats.json"), 'w') as f:
    json.dump(stats, f, indent=2, default=str)
print(f"✅ eda_final_stats.json")

# Save user-item pair analysis (will be useful for label design)
user_item.to_parquet(os.path.join(OUT_DIR, "user_item_pairs_raw.parquet"), index=False)
print(f"✅ user_item_pairs_raw.parquet: {len(user_item):,} pairs")

print(f"\n✅ FULL EDA COMPLETE — events df is FINAL")
print(f"   Next: Step 2 (label design + temporal split + save train/val/test)")


  0. APPLY SESSION THRESHOLD (30 min)
Session threshold: 20 min
Sessions: 234,189
Avg events/session: 3.2
Columns: ['timestamp', 'visitorid', 'event', 'itemid', 'transactionid', 'timestamp_dt', 'date', 'hour', 'day_of_week', 'year_month', 'week', 'session_id', 'session_position', 'session_length', 'session_unique_items', 'session_has_purchase', 'session_has_cart', 'session_duration_sec', 'time_in_session_sec']

  1. BASIC STATS — FINAL CLEAN DATA

  Events:          739,445
  Users:            65,559
  Items:            35,099
  Density:          0.0321%
  
  Views:           683,011 (92.4%)
  AddToCart:        40,877 (5.5%)
  Transactions:     15,557 (2.1%)
  Buyers:            7,350 (11.2% of users)
  Sessions:        234,189


  2. CONVERSION FUNNEL
  View → Cart:        5.98%
  Cart → Transaction: 38.06%
  View → Transaction: 2.28%

  3. USER-ITEM PAIR OUTCOMES
  Total user-item pairs: 387,343
  Purchased:           14,479 (3.74%)
  Carted only:         21,945 (5.67%)
  Viewed onl

C:\Users\NITRO V 15\AppData\Local\Temp\ipykernel_20844\1700651757.py:313: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  gini = 1 - 2 * np.trapz(c_arr, n_arr)


💾 outputs/figures/eda_02_items.png

  6. SESSION ANALYSIS
  Total sessions: 234,189
  Purchase sessions: 9,816 (4.19%)
  Cart sessions: 22,909 (9.78%)

  Session length (events):
    P25: 1
    P50: 2
    P75: 4
    P90: 7

  Session items:
    P50: 1
    P75: 2
    P90: 5

  Session duration (active sessions, n=119,894):
    P50: 367s (6.1min)
    P75: 849s (14.2min)
    P90: 1477s (24.6min)

  Purchase sessions avg events: 8.1
  Non-purchase sessions avg events: 2.9
  → Purchase sessions are longer → session_length is a feature
💾 outputs/figures/eda_03_sessions.png

  7. TEMPORAL ANALYSIS
  Daily events: mean=5320, median=5261
  Daily conversion rate: mean=2.091%

  Peak hours: [20, 21, 19]
  Peak days: ['Tue', 'Mon', 'Wed']

  Conversion rate by hour (top 5):
    17:00 → 2.99%
    18:00 → 2.91%
    19:00 → 2.81%
    16:00 → 2.71%
    21:00 → 2.70%
💾 outputs/figures/eda_04_temporal.png

  8. TEMPORAL SPLIT (70/15/15)
  Train cutoff: 2015-07-29
  Val cutoff:   2015-08-22
  Train:    5

# Step 3: Label, Weight and Split

In [16]:
"""
STEP 2: LABEL DESIGN + TEMPORAL SPLIT
Assumes: events df FINAL trong memory (from step1e)

Làm 3 việc:
  1. Mỗi (user, item) pair → assign label + weight
  2. Split theo thời gian 70/15/15
  3. Save train/val/test parquets

Output:
  data/processed/train_pairs.parquet
  data/processed/val_pairs.parquet
  data/processed/test_pairs.parquet
  data/processed/user_item_labeled.parquet (full)
"""

import pandas as pd
import numpy as np
import json
import os

OUT_DIR = "data/processed"
os.makedirs(OUT_DIR, exist_ok=True)

def section(title):
    print(f"\n{'='*70}")
    print(f"  {title}")
    print(f"{'='*70}")

if 'timestamp_dt' not in events.columns:
    events['timestamp_dt'] = pd.to_datetime(events['timestamp'], unit='ms')


# ============================================================
# 1. BUILD USER-ITEM PAIRS
# ============================================================
section("1. BUILD USER-ITEM PAIRS")

"""
Mỗi (visitorid, itemid) = 1 pair.
1 pair có thể có nhiều events (user view item 3 lần, cart 1 lần, mua 1 lần).
Aggregate: lấy highest action làm label.

Tại sao aggregate thay vì event-level labels:
  - Ranking model predict "user này có mua item này không?" = pair-level question
  - Nếu event-level: user view item 3 lần → 3 negative rows, 
    rồi mua → 1 positive row. Model confused: cùng (user, item) 
    vừa negative vừa positive.
  - Pair-level: view 3 lần rồi mua → label=1, clean.
"""

user_item = events.groupby(['visitorid', 'itemid']).agg(
    n_views=('event', lambda x: (x == 'view').sum()),
    n_carts=('event', lambda x: (x == 'addtocart').sum()),
    n_trans=('event', lambda x: (x == 'transaction').sum()),
    total_events=('event', 'count'),
    first_ts=('timestamp_dt', 'min'),
    last_ts=('timestamp_dt', 'max'),
    n_sessions=('session_id', 'nunique'),
).reset_index()

print(f"Total user-item pairs: {len(user_item):,}")


# ============================================================
# 2. ASSIGN LABELS + WEIGHTS
# ============================================================
section("2. ASSIGN LABELS + WEIGHTS")

"""
Label scheme:

  transaction    → label=1, weight=1.0
    Tại sao: User bỏ tiền mua = chắc chắn thích item.
    Đây là strongest signal trong implicit feedback.
    Weight 1.0 = fully confident.
  
  addtocart only → label=1, weight=0.5
    Tại sao: User bỏ vào giỏ = có intent mua.
    Cart abandonment (~62% từ EDA) có nhiều lý do không liên quan preference:
      - Giá shipping cao (thích item, ghét shipping)
      - Đợi giảm giá
      - Bị distract
    Nên cart = positive nhưng KHÔNG chắc chắn như purchase.
    Weight 0.5 = half confident.
    
  view only      → label=0, weight=0.7
    Tại sao: User ĐÃ THẤY item và KHÔNG action thêm.
    Đây là OBSERVED negative — quý hơn random negative.
    Nhưng weight không phải 1.0 vì:
      - EDA Part 3: avg 2.9 views trước khi mua → user có thể 
        đang ở view thứ 1-2, chưa quyết định
      - 47% users returning → có thể view hôm nay, mua tuần sau
    Weight 0.7 = khá confident là negative, nhưng không 100%.
    
  Tại sao KHÔNG có weight 0.1 cho "not seen" (random negatives):
    Ở bước này chỉ label OBSERVED interactions.
    Random negatives (items user chưa từng thấy) sẽ được sampling 
    ở Step 7 (retrieval) và Step 8 (ranking) khi cần.
    Mixing observed + random negatives trong cùng dataset → confuse model.
"""

def assign_label(row):
    if row['n_trans'] > 0:
        return 1, 1.0, 'transaction'
    elif row['n_carts'] > 0:
        return 1, 0.5, 'addtocart'
    else:
        return 0, 0.7, 'view_only'

labels = user_item.apply(assign_label, axis=1, result_type='expand')
user_item['label'] = labels[0]
user_item['weight'] = labels[1]
user_item['label_source'] = labels[2]

# Report
print("\nLabel distribution:")
for src in ['transaction', 'addtocart', 'view_only']:
    subset = user_item[user_item['label_source'] == src]
    n = len(subset)
    lbl = 1 if src != 'view_only' else 0
    w = {'transaction': 1.0, 'addtocart': 0.5, 'view_only': 0.7}[src]
    print(f"  {src:15s}: {n:>10,} ({n/len(user_item)*100:5.1f}%)  label={lbl}  weight={w}")

n_pos = (user_item['label'] == 1).sum()
n_neg = (user_item['label'] == 0).sum()
print(f"\n  Total positive (label=1): {n_pos:>10,} ({n_pos/len(user_item)*100:.1f}%)")
print(f"  Total negative (label=0): {n_neg:>10,} ({n_neg/len(user_item)*100:.1f}%)")
print(f"  Pos:Neg ratio = 1:{n_neg/max(n_pos,1):.0f}")


# ============================================================
# 3. TEMPORAL SPLIT
# ============================================================
section("3. TEMPORAL SPLIT (70/15/15)")

"""
Split dựa trên first_ts của mỗi pair (thời điểm user FIRST interact với item).

Tại sao first_ts thay vì last_ts:
  - User view item ngày 1, mua ngày 30.
  - Nếu dùng last_ts: pair rơi vào test (ngày 30).
    Nhưng training data biết user đã view item ngày 1 → leakage.
  - Nếu dùng first_ts: pair rơi vào train (ngày 1).
    Model learn từ toàn bộ trajectory (view → mua) = đúng.

Tại sao 70/15/15 thay vì 80/10/10:
  Data 137 ngày. 10% = ~14 ngày, 15% = ~20 ngày.
  20 ngày cho val/test = nhiều scenarios hơn, evaluation robust hơn.
"""

train_cutoff = user_item['first_ts'].quantile(0.70)
val_cutoff = user_item['first_ts'].quantile(0.85)

user_item['split'] = 'train'
user_item.loc[user_item['first_ts'] >= train_cutoff, 'split'] = 'val'
user_item.loc[user_item['first_ts'] >= val_cutoff, 'split'] = 'test'

print(f"Train cutoff: {train_cutoff.date()}")
print(f"Val cutoff:   {val_cutoff.date()}")

for split in ['train', 'val', 'test']:
    s = user_item[user_item['split'] == split]
    n = len(s)
    n_pos = (s['label'] == 1).sum()
    n_neg = (s['label'] == 0).sum()
    print(f"\n  {split:5s}: {n:>10,} pairs ({n/len(user_item)*100:.1f}%)")
    print(f"         Positive: {n_pos:>8,} ({n_pos/max(n,1)*100:.1f}%)")
    print(f"         Negative: {n_neg:>8,} ({n_neg/max(n,1)*100:.1f}%)")
    
    # Label source breakdown per split
    for src in ['transaction', 'addtocart', 'view_only']:
        n_src = (s['label_source'] == src).sum()
        print(f"         {src}: {n_src:>8,}")


# ============================================================
# 4. WARM/COLD ANALYSIS
# ============================================================
section("4. WARM/COLD ANALYSIS")

"""
Warm user = user xuất hiện trong training data → có embeddings + features
Cold user = user chỉ xuất hiện trong val/test → chỉ có popular items fallback

Tại sao quan trọng:
  - CF models (LightGCN, SASRec) CHỈ work cho warm users
  - Cold users = phải dùng popularity + content features
  - Phải report metrics RIÊNG cho warm vs cold
"""

train_users = set(user_item[user_item['split'] == 'train']['visitorid'].unique())
train_items = set(user_item[user_item['split'] == 'train']['itemid'].unique())

for split in ['val', 'test']:
    s = user_item[user_item['split'] == split]
    n = len(s)
    
    warm_both = s[s['visitorid'].isin(train_users) & s['itemid'].isin(train_items)]
    warm_user_cold_item = s[s['visitorid'].isin(train_users) & ~s['itemid'].isin(train_items)]
    cold_user_warm_item = s[~s['visitorid'].isin(train_users) & s['itemid'].isin(train_items)]
    cold_both = s[~s['visitorid'].isin(train_users) & ~s['itemid'].isin(train_items)]
    
    print(f"\n  {split}:")
    print(f"    Warm user + Warm item: {len(warm_both):>8,} ({len(warm_both)/max(n,1)*100:.1f}%) ← CF eval target")
    print(f"    Warm user + Cold item: {len(warm_user_cold_item):>8,} ({len(warm_user_cold_item)/max(n,1)*100:.1f}%)")
    print(f"    Cold user + Warm item: {len(cold_user_warm_item):>8,} ({len(cold_user_warm_item)/max(n,1)*100:.1f}%)")
    print(f"    Cold user + Cold item: {len(cold_both):>8,} ({len(cold_both)/max(n,1)*100:.1f}%)")
    
    # Positives in warm-warm (what CF can actually help with)
    ww_pos = (warm_both['label'] == 1).sum()
    print(f"    Warm-warm positives: {ww_pos:,} (CF has {ww_pos} positive pairs to evaluate on)")


# ============================================================
# 5. SAVE
# ============================================================
section("5. SAVE")

# Save full labeled data
user_item.to_parquet(os.path.join(OUT_DIR, "user_item_labeled.parquet"), index=False)
print(f"✅ user_item_labeled.parquet: {len(user_item):,} pairs")

# Save splits separately
for split in ['train', 'val', 'test']:
    s = user_item[user_item['split'] == split]
    s.to_parquet(os.path.join(OUT_DIR, f"{split}_pairs.parquet"), index=False)
    print(f"✅ {split}_pairs.parquet: {len(s):,} pairs")

# Save split metadata
split_stats = {
    'train_cutoff': str(train_cutoff),
    'val_cutoff': str(val_cutoff),
    'train_pairs': int(len(user_item[user_item['split'] == 'train'])),
    'val_pairs': int(len(user_item[user_item['split'] == 'val'])),
    'test_pairs': int(len(user_item[user_item['split'] == 'test'])),
    'n_positive': int(n_pos),
    'n_negative': int(n_neg),
    'train_users': int(len(train_users)),
    'train_items': int(len(train_items)),
}
with open(os.path.join(OUT_DIR, "step2_split_stats.json"), 'w') as f:
    json.dump(split_stats, f, indent=2, default=str)
print(f"✅ step2_split_stats.json")

print(f"""
╔═══════════════════════════════════════════════╗
║  STEP 2 COMPLETE                              ║
╠═══════════════════════════════════════════════╣
║  Labels: transaction=1(w=1.0)                 ║
║          addtocart=1(w=0.5)                   ║
║          view_only=0(w=0.7)                   ║
║  Split: 70/15/15 temporal on first_ts         ║
║                                               ║
║  Next: Step 3 (Feature Engineering)           ║
║    needs: events df + user_item_labeled       ║
╚═══════════════════════════════════════════════╝
""")


  1. BUILD USER-ITEM PAIRS
Total user-item pairs: 387,343

  2. ASSIGN LABELS + WEIGHTS

Label distribution:
  transaction    :     14,479 (  3.7%)  label=1  weight=1.0
  addtocart      :     21,945 (  5.7%)  label=1  weight=0.5
  view_only      :    350,919 ( 90.6%)  label=0  weight=0.7

  Total positive (label=1):     36,424 (9.4%)
  Total negative (label=0):    350,919 (90.6%)
  Pos:Neg ratio = 1:10

  3. TEMPORAL SPLIT (70/15/15)
Train cutoff: 2015-07-28
Val cutoff:   2015-08-21

  train:    271,140 pairs (70.0%)
         Positive:   24,118 (8.9%)
         Negative:  247,022 (91.1%)
         transaction:    9,565
         addtocart:   14,553
         view_only:  247,022

  val  :     58,101 pairs (15.0%)
         Positive:    6,056 (10.4%)
         Negative:   52,045 (89.6%)
         transaction:    2,422
         addtocart:    3,634
         view_only:   52,045

  test :     58,102 pairs (15.0%)
         Positive:    6,250 (10.8%)
         Negative:   51,852 (89.2%)
         tran

# Step 4: Feature Engineering

In [47]:
"""
STEP 3 v4: FEATURES — LEAK-FREE WITH LEAVE-ONE-OUT SIGNALS
Fix: v3 quá strict (remove ALL cart/trans) → no signal → AUC < 0.5
Solution: dùng cart/trans signals nhưng LEAVE-ONE-OUT:
  - item_trans_count cho pair (user_A, item_X) = item X's trans count FROM OTHER USERS (not user_A)
  - user_trans_count cho pair (user_A, item_X) = user A's trans count ON OTHER ITEMS (not item_X)
"""

import pandas as pd
import numpy as np
import json
import os

OUT_DIR = "data/processed"
FEAT_DIR = "data/processed/features"
os.makedirs(FEAT_DIR, exist_ok=True)

def section(title):
    print(f"\n{'='*70}")
    print(f"  {title}")
    print(f"{'='*70}")

if 'timestamp_dt' not in events.columns:
    events['timestamp_dt'] = pd.to_datetime(events['timestamp'], unit='ms')

user_item = pd.read_parquet(os.path.join(OUT_DIR, "user_item_labeled.parquet"))
train_cutoff = user_item['first_ts'].quantile(0.70)
train_events = events[events['timestamp_dt'] <= train_cutoff].copy()
train_views = train_events[train_events['event'] == 'view']

print(f"Train cutoff: {train_cutoff}")
print(f"Training events: {len(train_events):,}")


# ============================================================
# A. USER FEATURES
# ============================================================
section("A. USER FEATURES")

"""
View-based features (safe) + GLOBAL user buyer tendency.
user_n_trans_global = total transactions by this user across ALL items.
This is NOT leakage because:
  - For pair (user_A, item_X): user_A bought items Y, Z, W
  - Feature says "user_A is a buyer" — not "user_A bought item_X"
  - Same feature value for ALL items for user_A
  
HOWEVER: for pairs where user_A DID buy item_X, this feature includes
that transaction. To be strict, we use leave-one-out in cross features.
For user-level global features, the single item's contribution is tiny
(user bought 5 items, removing 1 → 4, still a buyer).
"""

user_feat = train_events.groupby('visitorid').agg(
    user_total_events=('event', 'count'),
    user_n_views=('event', lambda x: (x == 'view').sum()),
    user_n_carts_global=('event', lambda x: (x == 'addtocart').sum()),
    user_n_trans_global=('event', lambda x: (x == 'transaction').sum()),
    user_unique_items=('itemid', 'nunique'),
    user_n_sessions=('session_id', 'nunique'),
    user_avg_session_length=('session_length', 'mean'),
    user_avg_session_duration=('session_duration_sec', 'mean'),
    user_first_event=('timestamp_dt', 'min'),
    user_last_event=('timestamp_dt', 'max'),
).reset_index()

user_feat['user_is_buyer'] = (user_feat['user_n_trans_global'] > 0).astype(int)
user_feat['user_is_carter'] = (user_feat['user_n_carts_global'] > 0).astype(int)
user_feat['user_events_per_session'] = (
    user_feat['user_total_events'] / user_feat['user_n_sessions'].clip(lower=1)
)
user_feat['user_lifespan_days'] = (
    user_feat['user_last_event'] - user_feat['user_first_event']
).dt.days
user_feat['user_recency_days'] = (train_cutoff - user_feat['user_last_event']).dt.days

# View diversity
user_vd = train_views.groupby('visitorid').agg(
    user_avg_views_per_item=('itemid', lambda x: len(x) / max(x.nunique(), 1)),
).reset_index()
user_feat = user_feat.merge(user_vd, on='visitorid', how='left')

user_feat = user_feat.drop(columns=['user_first_event', 'user_last_event'])
user_feat = user_feat.fillna(0)

print(f"User features: {len(user_feat):,} users × {len(user_feat.columns)-1} features")
for col in user_feat.columns:
    if col != 'visitorid':
        print(f"  {col:35s}  median={user_feat[col].median():.2f}")


# ============================================================
# B. ITEM FEATURES
# ============================================================
section("B. ITEM FEATURES")

"""
Item popularity includes transaction count FROM ALL USERS.
item_n_trans_global = how many transactions this item received from ANY user.
NOT leakage because: for pair (user_A, item_X), other users' purchases 
of item_X are independent of user_A's label.

Bayesian smoothed conversion rate from ALL users' data.
"""

item_feat = train_events.groupby('itemid').agg(
    item_total_events=('event', 'count'),
    item_n_views=('event', lambda x: (x == 'view').sum()),
    item_n_carts_global=('event', lambda x: (x == 'addtocart').sum()),
    item_n_trans_global=('event', lambda x: (x == 'transaction').sum()),
    item_unique_visitors=('visitorid', 'nunique'),
).reset_index()

# Bayesian smoothed rates (from ALL users)
ALPHA, BETA = 1, 50
item_feat['item_conversion_rate'] = (
    (item_feat['item_n_trans_global'] + ALPHA) /
    (item_feat['item_n_views'] + ALPHA + BETA)
)
item_feat['item_cart_rate'] = (
    (item_feat['item_n_carts_global'] + ALPHA) /
    (item_feat['item_n_views'] + ALPHA + BETA)
)
item_feat['item_views_per_visitor'] = (
    item_feat['item_n_views'] / item_feat['item_unique_visitors'].clip(lower=1)
)

# Temporal
item_temporal = train_events.groupby('itemid')['timestamp_dt'].agg(['min', 'max']).reset_index()
item_temporal.columns = ['itemid', '_first', '_last']
item_feat = item_feat.merge(item_temporal, on='itemid', how='left')
item_feat['item_age_days'] = (train_cutoff - item_feat['_first']).dt.days
item_feat['item_recency_days'] = (train_cutoff - item_feat['_last']).dt.days
item_feat = item_feat.drop(columns=['_first', '_last'])

# Category
try:
    props = pd.concat([
        pd.read_csv("data/raw/item_properties_part1.csv", dtype=str),
        pd.read_csv("data/raw/item_properties_part2.csv", dtype=str),
    ])
    cat_props = props[props['property'] == 'categoryid'].copy()
    cat_props['timestamp'] = pd.to_numeric(cat_props['timestamp'])
    cat_props['ts_dt'] = pd.to_datetime(cat_props['timestamp'], unit='ms')
    cat_props = cat_props[cat_props['ts_dt'] <= train_cutoff]
    cat_props = cat_props.sort_values('timestamp').drop_duplicates('itemid', keep='last')
    item_category = cat_props[['itemid', 'value']].rename(columns={'value': 'categoryid'})
    item_feat = item_feat.merge(item_category, on='itemid', how='left')
    item_feat['categoryid'] = item_feat['categoryid'].fillna('unknown').astype(str)
    del props
except:
    item_feat['categoryid'] = 'unknown'

item_feat = item_feat.fillna(0)

print(f"Item features: {len(item_feat):,} items × {len(item_feat.columns)-1} features")
for col in item_feat.columns:
    if col not in ['itemid', 'categoryid']:
        print(f"  {col:35s}  median={item_feat[col].median():.2f}")


# ============================================================
# C. CROSS FEATURES — LEAVE-ONE-OUT
# ============================================================
section("C. CROSS FEATURES (leave-one-out)")

"""
CRITICAL: cross features cho pair (user_A, item_X) must EXCLUDE
events of that exact pair.

cross_cat_trans_loo: user_A's transactions in item_X's category,
  EXCLUDING item_X itself. "User bought other items in this category."
  
cross_cat_carts_loo: same but for carts.

This is NOT leakage because:
  - user_A buying item_Y (same category as X) is independent of
    whether user_A buys item_X
  - It tells model "user likes this category" without revealing
    "user bought this specific item"
"""

# Step 1: Get category for each item
ui = user_item[['visitorid', 'itemid', 'label', 'weight', 'split']].copy()
ui = ui.merge(item_feat[['itemid', 'categoryid']], on='itemid', how='left')
ui['categoryid'] = ui['categoryid'].fillna('unknown')

# Step 2: User-item view history
train_ui_views = train_views.groupby(['visitorid', 'itemid']).agg(
    cross_views_this_item=('event', 'count'),
).reset_index()

train_ui_sessions = train_events.groupby(['visitorid', 'itemid']).agg(
    cross_sessions_with_item=('session_id', 'nunique'),
).reset_index()

# Step 3: User-category history (ALL event types) — will do LOO
train_with_cat = train_events.merge(
    item_feat[['itemid', 'categoryid']], on='itemid', how='left'
)

user_cat_full = train_with_cat.groupby(['visitorid', 'categoryid']).agg(
    cat_views_total=('event', lambda x: (x == 'view').sum()),
    cat_carts_total=('event', lambda x: (x == 'addtocart').sum()),
    cat_trans_total=('event', lambda x: (x == 'transaction').sum()),
    cat_unique_items=('itemid', 'nunique'),
).reset_index()

# Step 4: User-item pair-level counts (to subtract for LOO)
train_ui_pair = train_events.groupby(['visitorid', 'itemid']).agg(
    pair_views=('event', lambda x: (x == 'view').sum()),
    pair_carts=('event', lambda x: (x == 'addtocart').sum()),
    pair_trans=('event', lambda x: (x == 'transaction').sum()),
).reset_index()

# Merge everything into ui
ui = ui.merge(train_ui_views, on=['visitorid', 'itemid'], how='left')
ui = ui.merge(train_ui_sessions, on=['visitorid', 'itemid'], how='left')

ui = ui.merge(
    user_cat_full, on=['visitorid', 'categoryid'], how='left'
)
ui = ui.merge(
    train_ui_pair, on=['visitorid', 'itemid'], how='left'
)

# Fill NaN
for col in ['cross_views_this_item', 'cross_sessions_with_item',
            'cat_views_total', 'cat_carts_total', 'cat_trans_total', 'cat_unique_items',
            'pair_views', 'pair_carts', 'pair_trans']:
    ui[col] = ui[col].fillna(0)

# Step 5: LEAVE-ONE-OUT — subtract THIS pair's contribution from category totals
ui['cross_cat_views_loo'] = (ui['cat_views_total'] - ui['pair_views']).clip(lower=0)
ui['cross_cat_carts_loo'] = (ui['cat_carts_total'] - ui['pair_carts']).clip(lower=0)
ui['cross_cat_trans_loo'] = (ui['cat_trans_total'] - ui['pair_trans']).clip(lower=0)
ui['cross_cat_items_loo'] = (ui['cat_unique_items'] - 1).clip(lower=0)

# Category affinity LOO
ui['cross_cat_affinity_loo'] = (
    ui['cross_cat_trans_loo'] / ui['cross_cat_views_loo'].clip(lower=1)
)

# View intensity (safe — views only)
cat_views_per_user = ui.groupby('visitorid')['cat_views_total'].transform('sum').clip(lower=1)
ui['cross_cat_view_intensity'] = ui['cat_views_total'] / cat_views_per_user

# Drop intermediate columns
ui = ui.drop(columns=['cat_views_total', 'cat_carts_total', 'cat_trans_total',
                       'cat_unique_items', 'pair_views', 'pair_carts', 'pair_trans'])

cross_cols = [c for c in ui.columns if c.startswith('cross_')]
print(f"Cross features: {len(cross_cols)}")
for col in cross_cols:
    print(f"  {col:35s}  mean={ui[col].mean():.3f}")


# ============================================================
# D. CONTEXT FEATURES
# ============================================================
section("D. CONTEXT FEATURES")

first_events = events.sort_values('timestamp_dt').drop_duplicates(
    subset=['visitorid', 'itemid'], keep='first'
)[['visitorid', 'itemid', 'hour', 'day_of_week', 'session_position',
   'session_length', 'session_duration_sec', 'time_in_session_sec']]

first_events = first_events.rename(columns={
    'hour': 'ctx_hour', 'day_of_week': 'ctx_day_of_week',
    'session_position': 'ctx_session_position', 'session_length': 'ctx_session_length',
    'session_duration_sec': 'ctx_session_duration', 'time_in_session_sec': 'ctx_time_in_session',
})
first_events['ctx_is_weekend'] = first_events['ctx_day_of_week'].isin([5, 6]).astype(int)
first_events['ctx_time_bucket'] = first_events['ctx_hour'].apply(
    lambda h: 0 if 6 <= h < 12 else (1 if 12 <= h < 17 else (2 if 17 <= h < 22 else 3))
)

ui = ui.merge(first_events, on=['visitorid', 'itemid'], how='left')
ctx_cols = [c for c in ui.columns if c.startswith('ctx_')]
ui[ctx_cols] = ui[ctx_cols].fillna(0)


# ============================================================
# E. FINAL
# ============================================================
section("E. FINAL FEATURE TABLE")

ui = ui.merge(user_feat, on='visitorid', how='left')
item_feat_no_cat = item_feat.drop(columns=['categoryid'], errors='ignore')
ui = ui.merge(item_feat_no_cat, on='itemid', how='left')

numeric_cols = ui.select_dtypes(include=[np.number]).columns
ui[numeric_cols] = ui[numeric_cols].fillna(0)

drop_cols = [c for c in ui.columns if c.endswith('_x') or c.endswith('_y')]
if drop_cols:
    ui = ui.drop(columns=drop_cols)

id_cols = [c for c in ['visitorid', 'itemid', 'label', 'weight', 'split', 'categoryid']
           if c in ui.columns]
feature_cols = [c for c in ui.columns if c not in id_cols]

print(f"\nFinal: {len(ui):,} rows × {len(feature_cols)} features")
print(f"\nAll {len(feature_cols)} features:")
for i, col in enumerate(sorted(feature_cols)):
    print(f"  {i+1:3d}. {col}")

print(f"\nPer split:")
for split in ['train', 'val', 'test']:
    s = ui[ui['split'] == split]
    print(f"  {split:5s}: {len(s):>10,}  (pos:{(s['label']==1).sum():>6,})")


# ============================================================
# F. SAVE
# ============================================================
section("F. SAVE")

for col in ['categoryid']:
    if col in ui.columns:
        ui[col] = ui[col].astype(str)

ui.to_parquet(os.path.join(FEAT_DIR, "feature_table.parquet"), index=False)

feat_meta = {
    'id_columns': id_cols,
    'feature_columns': sorted(feature_cols),
    'n_features': len(feature_cols),
}
with open(os.path.join(FEAT_DIR, "feature_meta.json"), 'w') as f:
    json.dump(feat_meta, f, indent=2)

user_feat.to_parquet(os.path.join(FEAT_DIR, "user_features.parquet"), index=False)
for col in ['categoryid']:
    if col in item_feat.columns:
        item_feat[col] = item_feat[col].astype(str)
item_feat.to_parquet(os.path.join(FEAT_DIR, "item_features.parquet"), index=False)

print(f"✅ feature_table.parquet: {len(ui):,} rows × {len(feature_cols)} features")
print(f"\n✅ STEP 3 v4 COMPLETE — LEAK-FREE + LOO SIGNALS")

Train cutoff: 2015-07-28 17:57:13.282999808
Training events: 512,412

  A. USER FEATURES
User features: 48,087 users × 14 features
  user_total_events                    median=7.00
  user_n_views                         median=6.00
  user_n_carts_global                  median=0.00
  user_n_trans_global                  median=0.00
  user_unique_items                    median=4.00
  user_n_sessions                      median=2.00
  user_avg_session_length              median=4.75
  user_avg_session_duration            median=348.48
  user_is_buyer                        median=0.00
  user_is_carter                       median=0.00
  user_events_per_session              median=3.50
  user_lifespan_days                   median=0.00
  user_recency_days                    median=35.00
  user_avg_views_per_item              median=1.67

  B. ITEM FEATURES
Item features: 32,639 items × 11 features
  item_total_events                    median=8.00
  item_n_views                         

# Step 5: item2vec

In [24]:
"""
STEP 4: ITEM2VEC
Assumes: events df trong memory + data/processed/ files from step 2.

Item2Vec = Word2Vec on user view sequences.
Items co-occurring in same user's history → similar embeddings.

Output: data/processed/embeddings/item2vec_embeddings.npy

Cần: pip install gensim
"""

import pandas as pd
import numpy as np
import json
import os
from gensim.models import Word2Vec

OUT_DIR = "data/processed"
EMB_DIR = "data/processed/embeddings"
os.makedirs(EMB_DIR, exist_ok=True)

def section(title):
    print(f"\n{'='*70}")
    print(f"  {title}")
    print(f"{'='*70}")

if 'timestamp_dt' not in events.columns:
    events['timestamp_dt'] = pd.to_datetime(events['timestamp'], unit='ms')


# ============================================================
# 1. BUILD SEQUENCES (training data only)
# ============================================================
section("1. BUILD VIEW SEQUENCES")

"""
Dùng USER-LEVEL sequences (không phải session-level).

Tại sao: EDA Part 10 cho thấy:
  - User-level: median=7 items → đủ dài cho Word2Vec
  - Session-level: median=1 item → quá ngắn, Word2Vec cần ≥2 items

Sequence = list items user viewed, ordered by time.
Chỉ dùng training events (trước train_cutoff).
"""

train_pairs = pd.read_parquet(os.path.join(OUT_DIR, "train_pairs.parquet"))
# Dùng cùng cutoff với Step 2: 70th percentile first_ts
user_item_all = pd.read_parquet(os.path.join(OUT_DIR, "user_item_labeled.parquet"))
train_cutoff = user_item_all['first_ts'].quantile(0.70)
del user_item_all

train_views = events[
    (events['event'] == 'view') & (events['timestamp_dt'] <= train_cutoff)
].sort_values(['visitorid', 'timestamp_dt'])

# Build sequences: user → list of item IDs (strings)
sequences = []
for user, group in train_views.groupby('visitorid'):
    seq = [str(x) for x in group['itemid'].tolist()]
    if len(seq) >= 2:
        sequences.append(seq)

# Unique items in sequences
all_items = set()
for s in sequences:
    all_items.update(s)

print(f"Sequences: {len(sequences):,}")
print(f"Unique items: {len(all_items):,}")

seq_lens = [len(s) for s in sequences]
print(f"Sequence length: median={np.median(seq_lens):.0f}, "
      f"P75={np.percentile(seq_lens, 75):.0f}, "
      f"P90={np.percentile(seq_lens, 90):.0f}")


# ============================================================
# 2. TRAIN ITEM2VEC
# ============================================================
section("2. TRAIN")

"""
Hyperparameters:
  vector_size=128: item embedding dimension
  window=3: context window
    EDA session median=2 items → nearby items in sequence are within same session.
    Window=3 captures "items viewed close together".
  min_count=3: ignore very rare items
  sg=1: Skip-gram (better than CBOW for sparse data)
  negative=10: negative samples per positive pair
  ns_exponent=0.75: frequency smoothing (reduce popular item dominance)
  epochs=15: more than NLP default because sequences shorter
  workers=1: must be 1 in Jupyter notebooks
"""

model = Word2Vec(
    sentences=sequences,
    vector_size=128,
    window=3,
    min_count=3,
    sg=1,
    negative=10,
    ns_exponent=0.75,
    epochs=15,
    workers=1,
    seed=42,
)

vocab_size = len(model.wv)
print(f"Vocabulary: {vocab_size:,} items")

# Quick test
test_items = list(model.wv.index_to_key[:3])
for item in test_items:
    similar = model.wv.most_similar(item, topn=3)
    print(f"\n  Item {item} most similar:")
    for s_item, score in similar:
        print(f"    → {s_item} ({score:.3f})")


# ============================================================
# 3. SAVE
# ============================================================
section("3. SAVE")

item_ids = list(model.wv.index_to_key)
embeddings = np.array([model.wv[item] for item in item_ids])
item2emb_idx = {item: i for i, item in enumerate(item_ids)}

np.save(os.path.join(EMB_DIR, "item2vec_embeddings.npy"), embeddings)
with open(os.path.join(EMB_DIR, "item2vec_item2idx.json"), 'w') as f:
    json.dump(item2emb_idx, f)
model.save(os.path.join(EMB_DIR, "item2vec_model.bin"))

print(f"✅ item2vec_embeddings.npy: {embeddings.shape}")
print(f"✅ item2vec_item2idx.json: {len(item2emb_idx):,} items")
print(f"\n✅ STEP 4 COMPLETE")


  1. BUILD VIEW SEQUENCES
Sequences: 46,927
Unique items: 32,579
Sequence length: median=6, P75=10, P90=16

  2. TRAIN
Vocabulary: 29,909 items

  Item 257040 most similar:
    → 274947 (0.861)
    → 83197 (0.816)
    → 147763 (0.807)

  Item 309778 most similar:
    → 96876 (0.841)
    → 308553 (0.834)
    → 182107 (0.834)

  Item 461686 most similar:
    → 102131 (0.853)
    → 410102 (0.841)
    → 171878 (0.837)

  3. SAVE
✅ item2vec_embeddings.npy: (29909, 128)
✅ item2vec_item2idx.json: 29,909 items

✅ STEP 4 COMPLETE


# Step 6: LightGCN


In [27]:
"""
STEP 5: LIGHTGCN
Assumes: data/processed/ files from step 2.

Build user-item bipartite graph → propagate embeddings → learn collaborative signal.
Edge weights: view=0.1, cart=0.5, transaction=1.0

Output: data/processed/embeddings/lightgcn_user_emb.npy, lightgcn_item_emb.npy

Cần: pip install torch
"""

import pandas as pd
import numpy as np
import json
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

OUT_DIR = "data/processed"
EMB_DIR = "data/processed/embeddings"
os.makedirs(EMB_DIR, exist_ok=True)

def section(title):
    print(f"\n{'='*70}")
    print(f"  {title}")
    print(f"{'='*70}")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")


# ============================================================
# 1. BUILD GRAPH FROM TRAINING PAIRS
# ============================================================
section("1. BUILD GRAPH")

train_pairs = pd.read_parquet(os.path.join(OUT_DIR, "train_pairs.parquet"))

# Edge weights from label_source
weight_map = {'view_only': 0.1, 'addtocart': 0.5, 'transaction': 1.0}
train_pairs['edge_weight'] = train_pairs['label_source'].map(weight_map)

# ID mappings
unique_users = sorted(train_pairs['visitorid'].unique())
unique_items = sorted(train_pairs['itemid'].unique())
user2id = {u: i for i, u in enumerate(unique_users)}
item2id = {it: i for i, it in enumerate(unique_items)}

train_pairs['user_idx'] = train_pairs['visitorid'].map(user2id)
train_pairs['item_idx'] = train_pairs['itemid'].map(item2id)

n_users = len(user2id)
n_items = len(item2id)

print(f"Users: {n_users:,}, Items: {n_items:,}")
print(f"Edges: {len(train_pairs):,}")
for src, w in weight_map.items():
    n = (train_pairs['label_source'] == src).sum()
    print(f"  {src} (w={w}): {n:,}")

# Save mappings
with open(os.path.join(OUT_DIR, "user2id.json"), 'w') as f:
    json.dump(user2id, f)
with open(os.path.join(OUT_DIR, "item2id.json"), 'w') as f:
    json.dump(item2id, f)


# ============================================================
# 2. ADJACENCY MATRIX
# ============================================================
section("2. ADJACENCY MATRIX")

user_indices = torch.LongTensor(train_pairs['user_idx'].values)
item_indices = torch.LongTensor(train_pairs['item_idx'].values) + n_users
edge_weights = torch.FloatTensor(train_pairs['edge_weight'].values)

row = torch.cat([user_indices, item_indices])
col = torch.cat([item_indices, user_indices])
weights = torch.cat([edge_weights, edge_weights])

n_nodes = n_users + n_items

degree = torch.zeros(n_nodes)
degree.index_add_(0, row, weights)
degree_inv_sqrt = torch.pow(degree.clamp(min=1), -0.5)
norm_weights = degree_inv_sqrt[row] * weights * degree_inv_sqrt[col]

adj = torch.sparse_coo_tensor(
    torch.stack([row, col]), norm_weights, size=(n_nodes, n_nodes)
).coalesce().to(DEVICE)

print(f"Adjacency: {n_nodes} × {n_nodes}")


# ============================================================
# 3. MODEL + TRAINING
# ============================================================
section("3. TRAIN LIGHTGCN")

EMB_DIM = 64
N_LAYERS = 3
LR = 0.001
BATCH_SIZE = 2048
EPOCHS = 50
REG_WEIGHT = 1e-4

class LightGCN(nn.Module):
    def __init__(self, n_users, n_items, emb_dim, n_layers, adj):
        super().__init__()
        self.n_users = n_users
        self.n_items = n_items
        self.n_layers = n_layers
        self.adj = adj
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.item_emb = nn.Embedding(n_items, emb_dim)
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)
    
    def forward(self):
        all_emb = torch.cat([self.user_emb.weight, self.item_emb.weight], dim=0)
        emb_list = [all_emb]
        for _ in range(self.n_layers):
            all_emb = torch.sparse.mm(self.adj, all_emb)
            emb_list.append(all_emb)
        final_emb = torch.stack(emb_list, dim=0).mean(dim=0)
        return final_emb[:self.n_users], final_emb[self.n_users:]
    
    def bpr_loss(self, user_final, item_final, users, pos_items, neg_items):
        u_e = user_final[users]
        p_e = item_final[pos_items]
        n_e = item_final[neg_items]
        pos_scores = (u_e * p_e).sum(dim=1)
        neg_scores = (u_e * n_e).sum(dim=1)
        bpr = -torch.log(torch.sigmoid(pos_scores - neg_scores) + 1e-8).mean()
        reg = REG_WEIGHT * (
            self.user_emb.weight[users].norm(2).pow(2) +
            self.item_emb.weight[pos_items].norm(2).pow(2) +
            self.item_emb.weight[neg_items].norm(2).pow(2)
        ) / len(users)
        return bpr + reg

# Positive pairs for BPR
pos_pairs = train_pairs[train_pairs['label'] == 1][['user_idx', 'item_idx']].dropna()
pos_pairs = pos_pairs.astype(int)
user_pos_items = pos_pairs.groupby('user_idx')['item_idx'].apply(set).to_dict()

class BPRDataset(Dataset):
    def __init__(self, pos_pairs, n_items, user_pos_items):
        self.users = pos_pairs['user_idx'].values
        self.pos_items = pos_pairs['item_idx'].values
        self.n_items = n_items
        self.user_pos_items = user_pos_items
    def __len__(self):
        return len(self.users)
    def __getitem__(self, idx):
        user = self.users[idx]
        pos = self.pos_items[idx]
        neg = np.random.randint(0, self.n_items)
        while neg in self.user_pos_items.get(user, set()):
            neg = np.random.randint(0, self.n_items)
        return user, pos, neg

train_loader = DataLoader(
    BPRDataset(pos_pairs, n_items, user_pos_items),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=0
)

model = LightGCN(n_users, n_items, EMB_DIM, N_LAYERS, adj).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LR)

print(f"emb_dim={EMB_DIM}, layers={N_LAYERS}, epochs={EPOCHS}")
print(f"Positive pairs: {len(pos_pairs):,}, Batches: {len(train_loader):,}")

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    n_batches = 0
    user_final, item_final = model()
    
    for users, pos_items, neg_items in train_loader:
        users = users.to(DEVICE)
        pos_items = pos_items.to(DEVICE)
        neg_items = neg_items.to(DEVICE)
        loss = model.bpr_loss(user_final, item_final, users, pos_items, neg_items)
        optimizer.zero_grad()
        loss.backward(retain_graph=True)
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1
    
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"  Epoch {epoch+1:3d}/{EPOCHS}: loss = {total_loss/max(n_batches,1):.4f}")


# ============================================================
# 4. SAVE EMBEDDINGS
# ============================================================
section("4. SAVE")

model.eval()
with torch.no_grad():
    u_emb, i_emb = model()
    u_emb = u_emb.cpu().numpy()
    i_emb = i_emb.cpu().numpy()

np.save(os.path.join(EMB_DIR, "lightgcn_user_emb.npy"), u_emb)
np.save(os.path.join(EMB_DIR, "lightgcn_item_emb.npy"), i_emb)
torch.save(model.state_dict(), os.path.join(EMB_DIR, "lightgcn_model.pt"))

print(f"✅ lightgcn_user_emb.npy: {u_emb.shape}")
print(f"✅ lightgcn_item_emb.npy: {i_emb.shape}")
print(f"\n✅ STEP 5 COMPLETE")

Device: cpu

  1. BUILD GRAPH
Users: 48,087, Items: 32,639
Edges: 271,140
  view_only (w=0.1): 247,022
  addtocart (w=0.5): 14,553
  transaction (w=1.0): 9,565

  2. ADJACENCY MATRIX
Adjacency: 80726 × 80726

  3. TRAIN LIGHTGCN
emb_dim=64, layers=3, epochs=50
Positive pairs: 24,118, Batches: 12
  Epoch   1/50: loss = 0.6930
  Epoch  10/50: loss = 0.5841
  Epoch  20/50: loss = 0.3340
  Epoch  30/50: loss = 0.2026
  Epoch  40/50: loss = 0.1346
  Epoch  50/50: loss = 0.0958

  4. SAVE
✅ lightgcn_user_emb.npy: (48087, 64)
✅ lightgcn_item_emb.npy: (32639, 64)

✅ STEP 5 COMPLETE


# Step 7: SASREC


In [29]:
"""
STEP 6 v3: SASREC — FIXED nan val_loss
Main fixes:
  1. Filter out very short sequences that become all-padding
  2. Add epsilon to prevent log(0) in loss
  3. Skip all-padding batches in validation
"""

import pandas as pd
import numpy as np
import json
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

OUT_DIR = "data/processed"
EMB_DIR = "data/processed/embeddings"
os.makedirs(EMB_DIR, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

if 'timestamp_dt' not in events.columns:
    events['timestamp_dt'] = pd.to_datetime(events['timestamp'], unit='ms')

# ============================================================
# 1. BUILD SEQUENCES
# ============================================================
print("="*60)
print("  1. BUILD SEQUENCES")
print("="*60)

user_item_all = pd.read_parquet(os.path.join(OUT_DIR, "user_item_labeled.parquet"))
train_cutoff = user_item_all['first_ts'].quantile(0.70)
del user_item_all

train_events = events[events['timestamp_dt'] <= train_cutoff].sort_values(
    ['visitorid', 'timestamp_dt']
)

# Build sequences — ONLY strings, ONLY length >= 5 (need enough for input+target+context)
user_seqs = {}
for user, group in train_events.groupby('visitorid'):
    seq = [str(x) for x in group['itemid'].tolist()]
    if len(seq) >= 5:  # stricter filter: need meaningful sequences
        user_seqs[user] = seq

all_items = set()
for seq in user_seqs.values():
    all_items.update(seq)

item_list = sorted(all_items)
item2idx = {item: idx + 1 for idx, item in enumerate(item_list)}
n_items = len(item2idx) + 1

seq_lens = [len(s) for s in user_seqs.values()]
MAX_SEQ_LEN = min(int(np.percentile(seq_lens, 90)), 50)
EMB_DIM = 64
N_HEADS = 2
N_LAYERS = 2
DROPOUT = 0.2
LR = 0.001
BATCH_SIZE = 256
EPOCHS = 30

print(f"Users: {len(user_seqs):,}, Vocab: {n_items:,}")
print(f"Seq lengths: median={np.median(seq_lens):.0f}, P90={np.percentile(seq_lens, 90):.0f}")
print(f"MAX_SEQ_LEN={MAX_SEQ_LEN}")

# ============================================================
# 2. DATASET
# ============================================================
print("\n" + "="*60)
print("  2. DATASET")
print("="*60)

class SASRecDataset(Dataset):
    def __init__(self, sequences, item2idx, max_len):
        self.sequences = sequences
        self.item2idx = item2idx
        self.max_len = max_len
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        seq = self.sequences[idx]
        seq_idx = [self.item2idx.get(item, 0) for item in seq]
        
        # Truncate (keep most recent)
        if len(seq_idx) > self.max_len + 1:
            seq_idx = seq_idx[-(self.max_len + 1):]
        
        input_seq = seq_idx[:-1]
        target_seq = seq_idx[1:]
        
        # Left padding
        pad_len = self.max_len - len(input_seq)
        input_seq = [0] * pad_len + input_seq
        target_seq = [0] * pad_len + target_seq
        
        return torch.LongTensor(input_seq), torch.LongTensor(target_seq)

all_seqs = list(user_seqs.values())
np.random.seed(42)
indices = np.random.permutation(len(all_seqs))
split = int(0.9 * len(all_seqs))
train_seqs = [all_seqs[i] for i in indices[:split]]
val_seqs = [all_seqs[i] for i in indices[split:]]

train_loader = DataLoader(
    SASRecDataset(train_seqs, item2idx, MAX_SEQ_LEN),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=0
)
val_loader = DataLoader(
    SASRecDataset(val_seqs, item2idx, MAX_SEQ_LEN),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)
print(f"Train: {len(train_seqs):,}, Val: {len(val_seqs):,}")

# ============================================================
# 3. MODEL — simplified, more stable
# ============================================================
print("\n" + "="*60)
print("  3. MODEL")
print("="*60)

class SASRec(nn.Module):
    def __init__(self, n_items, emb_dim, max_len, n_heads, n_layers, dropout):
        super().__init__()
        self.item_emb = nn.Embedding(n_items, emb_dim, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, emb_dim)
        self.dropout = nn.Dropout(dropout)
        self.emb_norm = nn.LayerNorm(emb_dim)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=emb_dim, nhead=n_heads,
            dim_feedforward=emb_dim * 4, dropout=dropout,
            activation='gelu', batch_first=True,
            norm_first=True,  # Pre-LN = more stable training
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.output_proj = nn.Linear(emb_dim, n_items)
    
    def forward(self, input_seq):
        batch_size, seq_len = input_seq.shape
        
        # Embeddings
        item_e = self.item_emb(input_seq)
        positions = torch.arange(seq_len, device=input_seq.device).unsqueeze(0)
        pos_e = self.pos_emb(positions)
        seq_e = self.dropout(self.emb_norm(item_e + pos_e))
        
        # Causal mask — use additive float mask for compatibility
        causal_mask = torch.zeros(seq_len, seq_len, device=input_seq.device)
        causal_mask.masked_fill_(
            torch.triu(torch.ones(seq_len, seq_len, device=input_seq.device), diagonal=1).bool(),
            float('-inf')
        )
        
        # Padding mask
        padding_mask = (input_seq == 0)
        
        # If entire batch row is padding, skip to avoid nan
        # Replace all-padding rows with a dummy token to prevent nan
        all_pad_rows = padding_mask.all(dim=1)
        if all_pad_rows.any():
            input_seq_fixed = input_seq.clone()
            input_seq_fixed[all_pad_rows, -1] = 1  # dummy token
            padding_mask = (input_seq_fixed == 0)
            seq_e = self.dropout(self.emb_norm(
                self.item_emb(input_seq_fixed) + pos_e
            ))
        
        output = self.transformer(seq_e, mask=causal_mask, src_key_padding_mask=padding_mask)
        
        # Replace nan with 0 (safety net)
        output = torch.nan_to_num(output, nan=0.0)
        
        return self.output_proj(output)

model = SASRec(n_items, EMB_DIM, MAX_SEQ_LEN, N_HEADS, N_LAYERS, DROPOUT).to(DEVICE)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

# ============================================================
# 4. TRAIN
# ============================================================
print("\n" + "="*60)
print("  4. TRAIN")
print("="*60)

optimizer = optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss(ignore_index=0)

torch.save(model.state_dict(), os.path.join(EMB_DIR, "sasrec_best.pt"))
best_val_loss = float('inf')
patience = 5
patience_counter = 0

for epoch in range(EPOCHS):
    # --- Train ---
    model.train()
    train_loss = 0
    n_b = 0
    for inp, tgt in train_loader:
        inp, tgt = inp.to(DEVICE), tgt.to(DEVICE)
        logits = model(inp)
        loss = criterion(logits.view(-1, n_items), tgt.view(-1))
        
        if torch.isnan(loss):
            continue  # skip nan batches
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()
        n_b += 1
    
    # --- Validate ---
    model.eval()
    val_loss = 0
    v_b = 0
    with torch.no_grad():
        for inp, tgt in val_loader:
            inp, tgt = inp.to(DEVICE), tgt.to(DEVICE)
            logits = model(inp)
            loss = criterion(logits.view(-1, n_items), tgt.view(-1))
            
            if not torch.isnan(loss):
                val_loss += loss.item()
                v_b += 1
    
    avg_t = train_loss / max(n_b, 1)
    avg_v = val_loss / max(v_b, 1) if v_b > 0 else float('nan')
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"  Epoch {epoch+1:3d}/{EPOCHS}: train={avg_t:.4f}  val={avg_v:.4f}  (val_batches={v_b})")
    
    if not np.isnan(avg_v) and avg_v < best_val_loss:
        best_val_loss = avg_v
        patience_counter = 0
        torch.save(model.state_dict(), os.path.join(EMB_DIR, "sasrec_best.pt"))
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"  Early stopping at epoch {epoch+1}")
            break

model.load_state_dict(torch.load(os.path.join(EMB_DIR, "sasrec_best.pt"), map_location=DEVICE))
print(f"Best val loss: {best_val_loss:.4f}")

# ============================================================
# 5. SAVE
# ============================================================
print("\n" + "="*60)
print("  5. SAVE")
print("="*60)

model.eval()
with torch.no_grad():
    item_embeddings = model.item_emb.weight.cpu().numpy()[1:]

np.save(os.path.join(EMB_DIR, "sasrec_item_emb.npy"), item_embeddings)
torch.save(model.state_dict(), os.path.join(EMB_DIR, "sasrec_model.pt"))
with open(os.path.join(EMB_DIR, "sasrec_item2idx.json"), 'w') as f:
    json.dump(item2idx, f)

print(f"✅ sasrec_item_emb.npy: {item_embeddings.shape}")
print(f"✅ sasrec_model.pt")
print(f"\n✅ STEP 6 COMPLETE")

Device: cpu
  1. BUILD SEQUENCES
Users: 44,613, Vocab: 32,467
Seq lengths: median=7, P90=18
MAX_SEQ_LEN=18

  2. DATASET
Train: 40,151, Val: 4,462

  3. MODEL
Parameters: 4,289,491

  4. TRAIN


c:\Users\NITRO V 15\anaconda3\envs\recsys\lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
c:\Users\NITRO V 15\anaconda3\envs\recsys\lib\site-packages\torch\nn\functional.py:6041: UserWarning: Support for mismatched src_key_padding_mask and mask is deprecated. Use same type for both instead.
  warnings.warn(


  Epoch   1/30: train=10.2855  val=10.3152  (val_batches=18)
  Epoch   5/30: train=7.2969  val=10.1503  (val_batches=18)
  Epoch  10/30: train=5.0475  val=10.1443  (val_batches=18)
  Epoch  15/30: train=4.1131  val=10.1373  (val_batches=18)
  Epoch  20/30: train=3.5783  val=10.1406  (val_batches=18)
  Early stopping at epoch 21
Best val loss: 10.1372

  5. SAVE
✅ sasrec_item_emb.npy: (32466, 64)
✅ sasrec_model.pt

✅ STEP 6 COMPLETE


# Step 8: Retrieval

In [48]:
"""
STEP 7: TWO-TOWER + MULTI-CHANNEL RETRIEVAL
Input: embeddings from Steps 4/5/6, features from Step 3
Output: candidates per user + FAISS indices

4 retrieval channels:
  A. Two-Tower (features → embeddings → FAISS)  ← NEW
  B. LightGCN (graph collaborative → FAISS)
  C. Item2Vec (item similarity)
  D. Popular items (cold user fallback)

Cần: pip install faiss-cpu torch
"""

import pandas as pd
import numpy as np
import json
import os
import faiss
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

OUT_DIR = "data/processed"
EMB_DIR = "data/processed/embeddings"
FEAT_DIR = "data/processed/features"
RETR_DIR = "data/processed/retrieval"
os.makedirs(RETR_DIR, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def section(title):
    print(f"\n{'='*70}")
    print(f"  {title}")
    print(f"{'='*70}")


# ============================================================
# 0. LOAD DATA
# ============================================================
section("0. LOAD DATA")

# Embeddings
lightgcn_user_emb = np.load(os.path.join(EMB_DIR, "lightgcn_user_emb.npy"))
lightgcn_item_emb = np.load(os.path.join(EMB_DIR, "lightgcn_item_emb.npy"))
item2vec_emb = np.load(os.path.join(EMB_DIR, "item2vec_embeddings.npy"))

with open(os.path.join(OUT_DIR, "user2id.json")) as f:
    user2id = json.load(f)
with open(os.path.join(OUT_DIR, "item2id.json")) as f:
    item2id = json.load(f)
with open(os.path.join(EMB_DIR, "item2vec_item2idx.json")) as f:
    item2vec_item2idx = json.load(f)

id2item = {v: k for k, v in item2id.items()}
item2vec_idx2item = {v: k for k, v in item2vec_item2idx.items()}

# Features
user_feat = pd.read_parquet(os.path.join(FEAT_DIR, "user_features.parquet"))
item_feat = pd.read_parquet(os.path.join(FEAT_DIR, "item_features.parquet"))

# Pairs
train_pairs = pd.read_parquet(os.path.join(OUT_DIR, "train_pairs.parquet"))
val_pairs = pd.read_parquet(os.path.join(OUT_DIR, "val_pairs.parquet"))
test_pairs = pd.read_parquet(os.path.join(OUT_DIR, "test_pairs.parquet"))

# Events for recent items + popularity
if 'timestamp_dt' not in events.columns:
    events['timestamp_dt'] = pd.to_datetime(events['timestamp'], unit='ms')

user_item_all = pd.read_parquet(os.path.join(OUT_DIR, "user_item_labeled.parquet"))
train_cutoff = user_item_all['first_ts'].quantile(0.70)
del user_item_all
train_events = events[events['timestamp_dt'] <= train_cutoff]

print(f"LightGCN: user={lightgcn_user_emb.shape}, item={lightgcn_item_emb.shape}")
print(f"Item2Vec: {item2vec_emb.shape}")
print(f"Users: {len(user2id):,}, Items: {len(item2id):,}")


# ============================================================
# A. TWO-TOWER MODEL
# ============================================================
section("A. TWO-TOWER MODEL")

"""
Two-Tower:
  User Tower: user features → MLP → user vector (64-dim)
  Item Tower: item features → MLP → item vector (64-dim)
  Score = dot product(user_vec, item_vec)
  
Train: positive pairs score > negative pairs score (BPR loss)

Tại sao Two-Tower bổ sung LightGCN:
  - LightGCN: learn từ graph structure (indirect CF)
  - Two-Tower: learn từ features (behavioral patterns)
  - LightGCN tốt cho warm users, Two-Tower có thể handle warm users 
    với features KHÁC nhau (session behavior, category preference...)
"""

# Prepare features as tensors
user_feature_cols = [c for c in user_feat.columns if c != 'visitorid']
item_feature_cols = [c for c in item_feat.columns 
                     if c not in ['itemid', 'categoryid'] and item_feat[c].dtype in [np.float64, np.int64, np.float32]]

# User feature matrix (indexed by user2id)
user_feat_matrix = np.zeros((len(user2id), len(user_feature_cols)), dtype=np.float32)
user_feat_indexed = user_feat.set_index('visitorid')
for uid, idx in user2id.items():
    if uid in user_feat_indexed.index:
        user_feat_matrix[idx] = user_feat_indexed.loc[uid, user_feature_cols].values

# Item feature matrix (indexed by item2id)
item_feat_matrix = np.zeros((len(item2id), len(item_feature_cols)), dtype=np.float32)
item_feat_indexed = item_feat.set_index('itemid')
for iid, idx in item2id.items():
    if iid in item_feat_indexed.index:
        row = item_feat_indexed.loc[iid]
        item_feat_matrix[idx] = row[item_feature_cols].values

# Normalize features (zero mean, unit variance)
user_mean = user_feat_matrix.mean(axis=0, keepdims=True)
user_std = user_feat_matrix.std(axis=0, keepdims=True) + 1e-8
user_feat_matrix = (user_feat_matrix - user_mean) / user_std

item_mean = item_feat_matrix.mean(axis=0, keepdims=True)
item_std = item_feat_matrix.std(axis=0, keepdims=True) + 1e-8
item_feat_matrix = (item_feat_matrix - item_mean) / item_std

user_feat_tensor = torch.FloatTensor(user_feat_matrix).to(DEVICE)
item_feat_tensor = torch.FloatTensor(item_feat_matrix).to(DEVICE)

print(f"User features: {user_feat_matrix.shape}")
print(f"Item features: {item_feat_matrix.shape}")

# Two-Tower model
TOWER_DIM = 64

class TwoTower(nn.Module):
    def __init__(self, user_dim, item_dim, tower_dim):
        super().__init__()
        self.user_tower = nn.Sequential(
            nn.Linear(user_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, tower_dim),
        )
        self.item_tower = nn.Sequential(
            nn.Linear(item_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, tower_dim),
        )
    
    def forward(self, user_feats, item_feats):
        u = self.user_tower(user_feats)
        i = self.item_tower(item_feats)
        return u, i
    
    def score(self, u, i):
        return (u * i).sum(dim=1)

tt_model = TwoTower(len(user_feature_cols), len(item_feature_cols), TOWER_DIM).to(DEVICE)

# Train with BPR on positive pairs
pos_df = train_pairs[train_pairs['label'] == 1].copy()
pos_df['user_idx'] = pos_df['visitorid'].map(user2id)
pos_df['item_idx'] = pos_df['itemid'].map(item2id)
pos_df = pos_df.dropna(subset=['user_idx', 'item_idx']).astype({'user_idx': int, 'item_idx': int})
user_pos_items = pos_df.groupby('user_idx')['item_idx'].apply(set).to_dict()

n_items_tt = len(item2id)
optimizer = optim.Adam(tt_model.parameters(), lr=0.001)

print(f"Training Two-Tower... ({len(pos_df):,} positive pairs)")

for epoch in range(30):
    tt_model.train()
    total_loss = 0
    
    # Mini-batch BPR
    indices = np.random.permutation(len(pos_df))
    batch_size = 2048
    n_batches = 0
    
    for start in range(0, len(indices), batch_size):
        batch_idx = indices[start:start+batch_size]
        users = pos_df.iloc[batch_idx]['user_idx'].values
        pos_items = pos_df.iloc[batch_idx]['item_idx'].values
        neg_items = np.random.randint(0, n_items_tt, size=len(users))
        
        u_feat = user_feat_tensor[users]
        pi_feat = item_feat_tensor[pos_items]
        ni_feat = item_feat_tensor[neg_items]
        
        u_vec, pi_vec = tt_model(u_feat, pi_feat)
        _, ni_vec = tt_model(u_feat, ni_feat)
        
        pos_scores = tt_model.score(u_vec, pi_vec)
        neg_scores = tt_model.score(u_vec, ni_vec)
        
        loss = -torch.log(torch.sigmoid(pos_scores - neg_scores) + 1e-8).mean()
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1
    
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"  Epoch {epoch+1:3d}/30: loss = {total_loss/max(n_batches,1):.4f}")

# Generate Two-Tower embeddings for all users and items
tt_model.eval()
with torch.no_grad():
    tt_user_emb = tt_model.user_tower(user_feat_tensor).cpu().numpy()
    tt_item_emb = tt_model.item_tower(item_feat_tensor).cpu().numpy()

print(f"Two-Tower user emb: {tt_user_emb.shape}")
print(f"Two-Tower item emb: {tt_item_emb.shape}")

np.save(os.path.join(EMB_DIR, "twotower_user_emb.npy"), tt_user_emb)
np.save(os.path.join(EMB_DIR, "twotower_item_emb.npy"), tt_item_emb)
torch.save(tt_model.state_dict(), os.path.join(EMB_DIR, "twotower_model.pt"))


# ============================================================
# B. BUILD FAISS INDICES
# ============================================================
section("B. BUILD FAISS INDICES")

def normalize(emb):
    norms = np.linalg.norm(emb, axis=1, keepdims=True)
    return emb / np.clip(norms, 1e-8, None)

# Index 1: Two-Tower
tt_item_norm = normalize(tt_item_emb).astype(np.float32)
tt_user_norm = normalize(tt_user_emb).astype(np.float32)
index_tt = faiss.IndexFlatIP(TOWER_DIM)
index_tt.add(tt_item_norm)
print(f"Two-Tower FAISS: {index_tt.ntotal} items")

# Index 2: LightGCN
gcn_item_norm = normalize(lightgcn_item_emb).astype(np.float32)
gcn_user_norm = normalize(lightgcn_user_emb).astype(np.float32)
index_gcn = faiss.IndexFlatIP(lightgcn_item_emb.shape[1])
index_gcn.add(gcn_item_norm)
print(f"LightGCN FAISS: {index_gcn.ntotal} items")

# Index 3: Item2Vec
i2v_norm = normalize(item2vec_emb).astype(np.float32)
index_i2v = faiss.IndexFlatIP(item2vec_emb.shape[1])
index_i2v.add(i2v_norm)
print(f"Item2Vec FAISS: {index_i2v.ntotal} items")


# ============================================================
# C. POPULARITY CHANNEL
# ============================================================
section("C. POPULARITY")

item_pop = train_events.groupby('itemid')['visitorid'].nunique().sort_values(ascending=False)
top_popular = item_pop.head(100).index.tolist()
print(f"Top 100 popular items ready")


# ============================================================
# D. RETRIEVE CANDIDATES
# ============================================================
section("D. RETRIEVE CANDIDATES")

K_TT = 30    # Two-Tower candidates
K_GCN = 30   # LightGCN candidates
K_I2V = 20   # Item2Vec candidates
K_POP = 20   # Popular fallback

# User → train items (for filtering already seen)
user_train_items = train_pairs.groupby('visitorid')['itemid'].apply(set).to_dict()

# User → recent items (for Item2Vec seeds)
recent_views = train_events[train_events['event'] == 'view'].sort_values('timestamp_dt')
user_recent_items = recent_views.groupby('visitorid')['itemid'].apply(
    lambda x: list(x.tail(3))
).to_dict()

def retrieve_for_user(visitor_id):
    candidates = set()
    seen = user_train_items.get(visitor_id, set())
    
    # Channel A: Two-Tower
    if visitor_id in user2id:
        uid = user2id[visitor_id]
        vec = tt_user_norm[uid:uid+1]
        D, I = index_tt.search(vec, K_TT + len(seen))
        for idx in I[0]:
            if idx < len(id2item):
                iid = id2item[idx]
                if iid not in seen:
                    candidates.add(('tt', iid))
                    if sum(1 for c in candidates if c[0] == 'tt') >= K_TT:
                        break
    
    # Channel B: LightGCN
    if visitor_id in user2id:
        uid = user2id[visitor_id]
        vec = gcn_user_norm[uid:uid+1]
        D, I = index_gcn.search(vec, K_GCN + len(seen))
        for idx in I[0]:
            if idx < len(id2item):
                iid = id2item[idx]
                if iid not in seen:
                    candidates.add(('gcn', iid))
                    if sum(1 for c in candidates if c[0] == 'gcn') >= K_GCN:
                        break
    
    # Channel C: Item2Vec
    recent = user_recent_items.get(visitor_id, [])
    for seed in recent:
        if seed in item2vec_item2idx:
            sidx = item2vec_item2idx[seed]
            vec = i2v_norm[sidx:sidx+1]
            D, I = index_i2v.search(vec, K_I2V)
            for idx in I[0]:
                if idx in item2vec_idx2item:
                    iid = item2vec_idx2item[idx]
                    if iid not in seen and iid != seed:
                        candidates.add(('i2v', iid))
    
    # Channel D: Popular
    for iid in top_popular:
        if iid not in seen:
            candidates.add(('pop', iid))
            if sum(1 for c in candidates if c[0] == 'pop') >= K_POP:
                break
    
    # Return deduplicated items + track which channel
    item_channels = {}
    for channel, iid in candidates:
        if iid not in item_channels:
            item_channels[iid] = channel
    
    return item_channels

# Retrieve for val + test
print(f"Retrieving candidates...")

val_candidates = {}
for i, uid in enumerate(val_pairs['visitorid'].unique()):
    val_candidates[uid] = retrieve_for_user(uid)
    if (i+1) % 5000 == 0:
        print(f"  Val: {i+1:,}")

test_candidates = {}
for i, uid in enumerate(test_pairs['visitorid'].unique()):
    test_candidates[uid] = retrieve_for_user(uid)
    if (i+1) % 5000 == 0:
        print(f"  Test: {i+1:,}")

val_sizes = [len(c) for c in val_candidates.values()]
test_sizes = [len(c) for c in test_candidates.values()]
print(f"\nVal candidates/user: mean={np.mean(val_sizes):.0f}, median={np.median(val_sizes):.0f}")
print(f"Test candidates/user: mean={np.mean(test_sizes):.0f}, median={np.median(test_sizes):.0f}")


# ============================================================
# E. EVALUATE RETRIEVAL PER CHANNEL
# ============================================================
section("E. EVALUATE RETRIEVAL")

def eval_recall(pairs_df, candidates_dict, k=50):
    positives = pairs_df[pairs_df['label'] == 1].groupby('visitorid')['itemid'].apply(set).to_dict()
    
    recalls = []
    channel_hits = {'tt': 0, 'gcn': 0, 'i2v': 0, 'pop': 0}
    channel_total = {'tt': 0, 'gcn': 0, 'i2v': 0, 'pop': 0}
    
    for user, pos_items in positives.items():
        if user not in candidates_dict:
            continue
        cands = candidates_dict[user]
        cand_items = set(cands.keys())
        
        hits = pos_items & cand_items
        recalls.append(len(hits) / len(pos_items) if pos_items else 0)
        
        # Track which channel found hits
        for item in hits:
            ch = cands[item]
            channel_hits[ch] = channel_hits.get(ch, 0) + 1
        for item, ch in cands.items():
            channel_total[ch] = channel_total.get(ch, 0) + 1
    
    print(f"  Recall@{k}: {np.mean(recalls):.4f} (evaluated on {len(recalls):,} users)")
    print(f"  Channel contribution to hits:")
    total_hits = sum(channel_hits.values())
    for ch in ['tt', 'gcn', 'i2v', 'pop']:
        h = channel_hits.get(ch, 0)
        t = channel_total.get(ch, 0)
        print(f"    {ch:4s}: {h:>5,} hits / {t:>8,} candidates ({h/max(total_hits,1)*100:.1f}% of hits)")
    
    return np.mean(recalls)

print("Val:")
val_recall = eval_recall(val_pairs, val_candidates)
print("\nTest:")
test_recall = eval_recall(test_pairs, test_candidates)


# ============================================================
# F. SAVE
# ============================================================
section("F. SAVE")

# Save candidates
def save_candidates(cands_dict, name):
    rows = []
    for user, item_channels in cands_dict.items():
        for item, channel in item_channels.items():
            rows.append({'visitorid': user, 'itemid': item, 'channel': channel})
    df = pd.DataFrame(rows)
    df.to_parquet(os.path.join(RETR_DIR, f"{name}_candidates.parquet"), index=False)
    return df

val_cand_df = save_candidates(val_candidates, 'val')
test_cand_df = save_candidates(test_candidates, 'test')

faiss.write_index(index_tt, os.path.join(RETR_DIR, "faiss_twotower.index"))
faiss.write_index(index_gcn, os.path.join(RETR_DIR, "faiss_lightgcn.index"))
faiss.write_index(index_i2v, os.path.join(RETR_DIR, "faiss_item2vec.index"))

with open(os.path.join(RETR_DIR, "popular_items.json"), 'w') as f:
    json.dump(top_popular, f)

print(f"✅ val_candidates.parquet: {len(val_cand_df):,}")
print(f"✅ test_candidates.parquet: {len(test_cand_df):,}")
print(f"✅ FAISS indices saved")
print(f"\n✅ STEP 7 COMPLETE")


  0. LOAD DATA
LightGCN: user=(48087, 64), item=(32639, 64)
Item2Vec: (29909, 128)
Users: 48,087, Items: 32,639

  A. TWO-TOWER MODEL
User features: (48087, 14)
Item features: (32639, 10)
Training Two-Tower... (24,118 positive pairs)
  Epoch   1/30: loss = 0.6376
  Epoch  10/30: loss = 0.2459
  Epoch  20/30: loss = 0.2022
  Epoch  30/30: loss = 0.1792
Two-Tower user emb: (48087, 64)
Two-Tower item emb: (32639, 64)

  B. BUILD FAISS INDICES
Two-Tower FAISS: 32639 items
LightGCN FAISS: 32639 items
Item2Vec FAISS: 29909 items

  C. POPULARITY
Top 100 popular items ready

  D. RETRIEVE CANDIDATES
Retrieving candidates...
  Val: 5,000
  Val: 10,000
  Test: 5,000
  Test: 10,000

Val candidates/user: mean=45, median=20
Test candidates/user: mean=39, median=20

  E. EVALUATE RETRIEVAL
Val:
  Recall@50: 0.0445 (evaluated on 3,005 users)
  Channel contribution to hits:
    tt  :    12 hits /   15,061 candidates (5.0% of hits)
    gcn :    39 hits /   14,503 candidates (16.2% of hits)
    i2v : 

In [49]:
"""
STEP 8: RANKING MODEL — LightGBM
Input: feature_table from Step 3, embedding similarities from Steps 4/5/7
Output: scored + ranked recommendations

Cần: pip install lightgbm
"""

import pandas as pd
import numpy as np
import json
import os
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, log_loss

OUT_DIR = "data/processed"
FEAT_DIR = "data/processed/features"
EMB_DIR = "data/processed/embeddings"
RANK_DIR = "data/processed/ranking"
os.makedirs(RANK_DIR, exist_ok=True)

def section(title):
    print(f"\n{'='*70}")
    print(f"  {title}")
    print(f"{'='*70}")


# ============================================================
# 1. LOAD + ADD EMBEDDING FEATURES
# ============================================================
section("1. LOAD DATA + EMBEDDING FEATURES")

feat_df = pd.read_parquet(os.path.join(FEAT_DIR, "feature_table.parquet"))
with open(os.path.join(FEAT_DIR, "feature_meta.json")) as f:
    feat_meta = json.load(f)

feature_cols = list(feat_meta['feature_columns'])

# Add embedding dot product as feature
try:
    gcn_u = np.load(os.path.join(EMB_DIR, "lightgcn_user_emb.npy"))
    gcn_i = np.load(os.path.join(EMB_DIR, "lightgcn_item_emb.npy"))
    tt_u = np.load(os.path.join(EMB_DIR, "twotower_user_emb.npy"))
    tt_i = np.load(os.path.join(EMB_DIR, "twotower_item_emb.npy"))
    
    with open(os.path.join(OUT_DIR, "user2id.json")) as f:
        user2id = json.load(f)
    with open(os.path.join(OUT_DIR, "item2id.json")) as f:
        item2id = json.load(f)
    
    # Compute dot products
    gcn_dots = []
    tt_dots = []
    for _, row in feat_df.iterrows():
        uid = user2id.get(row['visitorid'])
        iid = item2id.get(row['itemid'])
        if uid is not None and iid is not None:
            gcn_dots.append(np.dot(gcn_u[uid], gcn_i[iid]))
            tt_dots.append(np.dot(tt_u[uid], tt_i[iid]))
        else:
            gcn_dots.append(0.0)
            tt_dots.append(0.0)
    
    feat_df['emb_lightgcn_dot'] = gcn_dots
    feat_df['emb_twotower_dot'] = tt_dots
    feature_cols.extend(['emb_lightgcn_dot', 'emb_twotower_dot'])
    print(f"  Added emb_lightgcn_dot + emb_twotower_dot")
except Exception as e:
    print(f"  Could not add embedding features: {e}")

print(f"Total features: {len(feature_cols)}")


# ============================================================
# 2. PREPARE SPLITS
# ============================================================
section("2. PREPARE SPLITS")

# Only numeric features
numeric_features = [c for c in feature_cols 
                    if feat_df[c].dtype in [np.float64, np.float32, np.int64, np.int32, np.bool_, np.uint8]]

train_df = feat_df[feat_df['split'] == 'train']
val_df = feat_df[feat_df['split'] == 'val']
test_df = feat_df[feat_df['split'] == 'test']

X_train, y_train, w_train = train_df[numeric_features].values, train_df['label'].values, train_df['weight'].values
X_val, y_val, w_val = val_df[numeric_features].values, val_df['label'].values, val_df['weight'].values
X_test, y_test = test_df[numeric_features].values, test_df['label'].values

pos_neg_ratio = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
print(f"Features: {len(numeric_features)}")
print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")
print(f"Pos:Neg = 1:{pos_neg_ratio:.0f}")


# ============================================================
# 3. TRAIN LIGHTGBM
# ============================================================
section("3. TRAIN")

params = {
    'objective': 'binary',
    'metric': 'auc',
    'scale_pos_weight': pos_neg_ratio,
    'num_leaves': 63,
    'learning_rate': 0.05,
    'max_depth': 7,
    'min_child_samples': 50,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'random_state': 42,
    'verbose': -1,
}

train_data = lgb.Dataset(X_train, label=y_train, weight=w_train, feature_name=numeric_features)
val_data = lgb.Dataset(X_val, label=y_val, weight=w_val, feature_name=numeric_features, reference=train_data)

model = lgb.train(
    params, train_data,
    valid_sets=[train_data, val_data],
    valid_names=['train', 'val'],
    num_boost_round=500,
    callbacks=[lgb.early_stopping(20), lgb.log_evaluation(50)],
)
print(f"Best iteration: {model.best_iteration}")


# ============================================================
# 4. EVALUATE
# ============================================================
section("4. EVALUATE")

val_pred = model.predict(X_val)
test_pred = model.predict(X_test)

val_auc = roc_auc_score(y_val, val_pred)
test_auc = roc_auc_score(y_test, test_pred)
print(f"Val  AUC: {val_auc:.4f}")
print(f"Test AUC: {test_auc:.4f}")

# NDCG@K per user
def ndcg_at_k(df, preds, k=20):
    df = df.copy()
    df['pred'] = preds
    scores = []
    for user, g in df.groupby('visitorid'):
        if g['label'].sum() == 0 or len(g) < 2:
            continue
        top_k = g.nlargest(k, 'pred')
        dcg = sum(top_k['label'].values[i] / np.log2(i+2) for i in range(len(top_k)))
        ideal = sorted(g['label'].values, reverse=True)[:k]
        idcg = sum(ideal[i] / np.log2(i+2) for i in range(len(ideal)))
        scores.append(dcg / max(idcg, 1e-8))
    return np.mean(scores) if scores else 0

val_ndcg = ndcg_at_k(val_df, val_pred)
test_ndcg = ndcg_at_k(test_df, test_pred)
print(f"Val  NDCG@20: {val_ndcg:.4f}")
print(f"Test NDCG@20: {test_ndcg:.4f}")

# Hit Rate@K
test_df_copy = test_df.copy()
test_df_copy['pred'] = test_pred
hit_rates = {}
for k in [5, 10, 20]:
    hits = []
    for user, g in test_df_copy.groupby('visitorid'):
        top = g.nlargest(k, 'pred')
        hits.append(1 if top['label'].sum() > 0 else 0)
    hit_rates[k] = np.mean(hits)
    print(f"Test Hit Rate@{k}: {hit_rates[k]:.4f}")

# Warm vs Cold breakdown
with open(os.path.join(OUT_DIR, "user2id.json")) as f:
    train_user_ids = set(json.load(f).keys())

print(f"\nWarm user metrics (users in training):")
warm_mask = test_df_copy['visitorid'].isin(train_user_ids)
if warm_mask.sum() > 0:
    warm_auc = roc_auc_score(y_test[warm_mask], test_pred[warm_mask])
    print(f"  AUC: {warm_auc:.4f}")
    warm_hits = []
    for user, g in test_df_copy[warm_mask].groupby('visitorid'):
        top = g.nlargest(20, 'pred')
        warm_hits.append(1 if top['label'].sum() > 0 else 0)
    print(f"  Hit Rate@20: {np.mean(warm_hits):.4f}")

print(f"\nCold user metrics:")
cold_mask = ~warm_mask
if cold_mask.sum() > 0 and y_test[cold_mask].sum() > 0:
    cold_auc = roc_auc_score(y_test[cold_mask], test_pred[cold_mask])
    print(f"  AUC: {cold_auc:.4f}")


# ============================================================
# 5. FEATURE IMPORTANCE
# ============================================================
section("5. FEATURE IMPORTANCE")

importance = model.feature_importance(importance_type='gain')
feat_imp = pd.DataFrame({'feature': numeric_features, 'importance': importance})
feat_imp = feat_imp.sort_values('importance', ascending=False)

print("Top 15 features:")
for _, row in feat_imp.head(15).iterrows():
    bar = '█' * max(1, int(row['importance'] / feat_imp['importance'].max() * 30))
    print(f"  {row['feature']:35s} {row['importance']:>10.0f} {bar}")

feat_imp.to_csv(os.path.join(RANK_DIR, "feature_importance.csv"), index=False)


# ============================================================
# 6. SAVE
# ============================================================
section("6. SAVE")

model.save_model(os.path.join(RANK_DIR, "lgbm_ranker.txt"))

test_df_copy[['visitorid', 'itemid', 'label', 'pred']].to_parquet(
    os.path.join(RANK_DIR, "test_predictions.parquet"), index=False)

stats = {
    'val_auc': round(val_auc, 4), 'test_auc': round(test_auc, 4),
    'val_ndcg20': round(val_ndcg, 4), 'test_ndcg20': round(test_ndcg, 4),
    'hit_rates': {str(k): round(v, 4) for k, v in hit_rates.items()},
    'n_features': len(numeric_features),
    'best_iteration': model.best_iteration,
}
with open(os.path.join(RANK_DIR, "ranking_stats.json"), 'w') as f:
    json.dump(stats, f, indent=2)

print(f"✅ lgbm_ranker.txt")
print(f"✅ test_predictions.parquet")
print(f"✅ ranking_stats.json")
print(f"\n✅ STEP 8 COMPLETE")


  1. LOAD DATA + EMBEDDING FEATURES
  Added emb_lightgcn_dot + emb_twotower_dot
Total features: 42

  2. PREPARE SPLITS
Features: 42
Train: 271,140 | Val: 58,101 | Test: 58,102
Pos:Neg = 1:10

  3. TRAIN
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[24]	train's auc: 0.993424	val's auc: 0.589594
Best iteration: 24

  4. EVALUATE
Val  AUC: 0.5873
Test AUC: 0.5638
Val  NDCG@20: 0.7466
Test NDCG@20: 0.7357
Test Hit Rate@5: 0.2175
Test Hit Rate@10: 0.2261
Test Hit Rate@20: 0.2280

Warm user metrics (users in training):
  AUC: 0.6359
  Hit Rate@20: 0.0994

Cold user metrics:
  AUC: 0.5393

  5. FEATURE IMPORTANCE
Top 15 features:
  user_is_carter                         1634083 ██████████████████████████████
  user_n_carts_global                     370537 ██████
  emb_twotower_dot                        221953 ████
  emb_lightgcn_dot                        196726 ███
  item_cart_rate                          136262 ██
  user_is_buyer      

In [51]:
"""
STEP 9+10: RE-RANKING + FINAL EVALUATION
Input: test_predictions from Step 8, embeddings from Steps 5/7
Output: final recommendations + full evaluation report
"""

import pandas as pd
import numpy as np
import json
import os

OUT_DIR = "data/processed"
EMB_DIR = "data/processed/embeddings"
RANK_DIR = "data/processed/ranking"
FINAL_DIR = "data/processed/final"
os.makedirs(FINAL_DIR, exist_ok=True)

def section(title):
    print(f"\n{'='*70}")
    print(f"  {title}")
    print(f"{'='*70}")


# ============================================================
# 1. LOAD
# ============================================================
section("1. LOAD")

test_pred = pd.read_parquet(os.path.join(RANK_DIR, "test_predictions.parquet"))

try:
    gcn_item = np.load(os.path.join(EMB_DIR, "lightgcn_item_emb.npy"))
    with open(os.path.join(OUT_DIR, "item2id.json")) as f:
        item2id = json.load(f)
    has_emb = True
except:
    has_emb = False

print(f"Test predictions: {len(test_pred):,}")
print(f"Embeddings: {'loaded' if has_emb else 'not available'}")


# ============================================================
# 2. MMR DIVERSITY RE-RANKING
# ============================================================
section("2. MMR RE-RANKING")

LAMBDA = 0.6  # balance relevance vs diversity
K = 20

def mmr_rerank(items, scores, embeddings, item2id_map, lam=LAMBDA, k=K):
    selected = []
    selected_embs = []
    remaining = list(range(len(items)))
    
    scores = np.array(scores)
    if scores.max() > scores.min():
        scores_norm = (scores - scores.min()) / (scores.max() - scores.min())
    else:
        scores_norm = np.ones_like(scores)
    
    for _ in range(min(k, len(items))):
        if not remaining:
            break
        best_idx = None
        best_mmr = -float('inf')
        
        for idx in remaining:
            rel = scores_norm[idx]
            max_sim = 0
            if selected_embs and items[idx] in item2id_map:
                ie = embeddings[item2id_map[items[idx]]]
                ie_norm = ie / (np.linalg.norm(ie) + 1e-8)
                for se in selected_embs:
                    sim = np.dot(ie_norm, se)
                    max_sim = max(max_sim, sim)
            
            mmr = lam * rel - (1 - lam) * max_sim
            if mmr > best_mmr:
                best_mmr = mmr
                best_idx = idx
        
        selected.append(best_idx)
        if items[best_idx] in item2id_map:
            e = embeddings[item2id_map[items[best_idx]]]
            selected_embs.append(e / (np.linalg.norm(e) + 1e-8))
        remaining.remove(best_idx)
    
    return [items[i] for i in selected]

# Apply MMR
final_recs = []
n_processed = 0

for user, group in test_pred.groupby('visitorid'):
    group = group.nlargest(min(50, len(group)), 'pred')
    items = group['itemid'].tolist()
    scores = group['pred'].tolist()
    labels = group.set_index('itemid')['label'].to_dict()
    
    if has_emb and len(items) > 1:
        reranked = mmr_rerank(items, scores, gcn_item, item2id)
    else:
        reranked = items[:K]
    
    for rank, item in enumerate(reranked[:K], 1):
        final_recs.append({
            'visitorid': user,
            'itemid': item,
            'final_rank': rank,
            'actual_label': labels.get(item, 0),
        })
    n_processed += 1

final_df = pd.DataFrame(final_recs)
print(f"Final recs: {len(final_df):,} for {final_df['visitorid'].nunique():,} users")


# ============================================================
# 3. FULL EVALUATION
# ============================================================
section("3. EVALUATION — Before vs After MMR")

def eval_metrics(recs_df, label_col='actual_label', rank_col='final_rank'):
    print(f"  {'K':>4s} {'HitRate':>8s} {'Precision':>10s}")
    results = {}
    for k in [5, 10, 20]:
        top_k = recs_df[recs_df[rank_col] <= k]
        hr = top_k.groupby('visitorid')[label_col].max().mean()
        prec = top_k.groupby('visitorid')[label_col].mean().mean()
        results[f'hr@{k}'] = round(hr, 4)
        results[f'prec@{k}'] = round(prec, 4)
        print(f"  @{k:2d}   {hr:.4f}    {prec:.4f}")
    return results

# Before MMR (pure ranking)
print("A. Pure Ranking (before MMR):")
pure_recs = []
for user, group in test_pred.groupby('visitorid'):
    top = group.nlargest(K, 'pred')
    for rank, (_, row) in enumerate(top.iterrows(), 1):
        pure_recs.append({
            'visitorid': user, 'itemid': row['itemid'],
            'final_rank': rank, 'actual_label': row['label'],
        })
pure_df = pd.DataFrame(pure_recs)
pure_results = eval_metrics(pure_df)

print("\nB. After MMR (λ=0.6):")
mmr_results = eval_metrics(final_df)

# Diversity comparison
if has_emb:
    def diversity(recs_df, k=10):
        divs = []
        for user, g in recs_df.groupby('visitorid'):
            items = g.head(k)['itemid'].tolist()
            embs = [gcn_item[item2id[i]] for i in items if i in item2id]
            if len(embs) >= 2:
                embs = np.array(embs)
                norms = np.linalg.norm(embs, axis=1, keepdims=True) + 1e-8
                embs = embs / norms
                sim = embs @ embs.T
                avg_sim = (sim.sum() - np.trace(sim)) / (len(embs) * (len(embs)-1))
                divs.append(1 - avg_sim)
        return np.mean(divs)
    
    div_pure = diversity(pure_df)
    div_mmr = diversity(final_df)
    print(f"\nDiversity@10:")
    print(f"  Pure: {div_pure:.4f}")
    print(f"  MMR:  {div_mmr:.4f} ({(div_mmr-div_pure)/max(div_pure,1e-8)*100:+.1f}%)")


# ============================================================
# 4. WARM vs COLD BREAKDOWN
# ============================================================
section("4. WARM vs COLD")

train_pairs = pd.read_parquet(os.path.join(OUT_DIR, "train_pairs.parquet"))
train_users = set(train_pairs['visitorid'].unique())

for name, mask_fn in [('Warm', lambda x: x.isin(train_users)),
                       ('Cold', lambda x: ~x.isin(train_users))]:
    subset = final_df[mask_fn(final_df['visitorid'])]
    n_users = subset['visitorid'].nunique()
    if n_users == 0:
        continue
    print(f"\n  {name} users ({n_users:,}):")
    for k in [5, 10, 20]:
        top = subset[subset['final_rank'] <= k]
        hr = top.groupby('visitorid')['actual_label'].max().mean()
        print(f"    Hit Rate@{k}: {hr:.4f}")


# ============================================================
# 5. SAVE
# ============================================================
section("5. SAVE")

final_df.to_parquet(os.path.join(FINAL_DIR, "final_recommendations.parquet"), index=False)

all_stats = {
    'mmr_lambda': LAMBDA,
    'pure_ranking': pure_results,
    'mmr_reranking': mmr_results,
}
if has_emb:
    all_stats['diversity_pure'] = round(div_pure, 4)
    all_stats['diversity_mmr'] = round(div_mmr, 4)

with open(os.path.join(FINAL_DIR, "final_stats.json"), 'w') as f:
    json.dump(all_stats, f, indent=2, default=lambda x: float(x) if hasattr(x, 'item') else str(x))

print(f"✅ final_recommendations.parquet")
print(f"✅ final_stats.json")
print(f"\n🎉 PIPELINE COMPLETE — 10 Steps")


  1. LOAD
Test predictions: 58,102
Embeddings: loaded

  2. MMR RE-RANKING
Final recs: 50,195 for 12,537 users

  3. EVALUATION — Before vs After MMR
A. Pure Ranking (before MMR):
     K  HitRate  Precision
  @ 5   0.2174    0.1228
  @10   0.2261    0.1218
  @20   0.2280    0.1217

B. After MMR (λ=0.6):
     K  HitRate  Precision
  @ 5   0.2179    0.1229
  @10   0.2262    0.1219
  @20   0.2280    0.1217

Diversity@10:
  Pure: 0.4934
  MMR:  0.4986 (+1.1%)

  4. WARM vs COLD

  Warm users (2,897):
    Hit Rate@5: 0.0946
    Hit Rate@10: 0.0994
    Hit Rate@20: 0.0994

  Cold users (9,640):
    Hit Rate@5: 0.2550
    Hit Rate@10: 0.2643
    Hit Rate@20: 0.2667

  5. SAVE
✅ final_recommendations.parquet
✅ final_stats.json

🎉 PIPELINE COMPLETE — 10 Steps
